In [ ]:
from pathlib import Path
import pandas as pd

cohorte = (
    Path(r"E:\Projet_Alzheimer\data\05_Manifestes")
    / "cohorte_359_20261024_184212_495491"
)
fichier = cohorte / "series_IRM_candidates.csv"

if not fichier.is_file():
    raise FileNotFoundError(f"Manifeste IRM introuvable : {fichier}")

# Lire les identifiants comme du texte pour conserver les zéros initiaux.
irm = pd.read_csv(fichier, sep=None, engine="python", dtype="string")
irm.columns = irm.columns.str.strip()

print("Fichier lu :", fichier.name)
print("Nombre de lignes :", len(irm), "(attendu : 2185)")
print("Colonnes :", " | ".join(irm.columns))

In [ ]:
# Utiliser les identifiants et la date déjà préparés dans le manifeste.
series = irm[["PTID", "date_irm", "Image ID"]].copy()
for col in series.columns:
    series[col] = series[col].str.strip().replace("", pd.NA)

if series.isna().any().any():
    raise ValueError("Une ligne n'a pas de participant, de date ou d'Image ID.")

series["date_irm"] = pd.to_datetime(series["date_irm"], errors="raise").dt.normalize()

# Une ligne du résultat = une visite d'un participant.
visites_irm = (
    series.groupby(["PTID", "date_irm"])["Image ID"]
    .nunique()
    .reset_index(name="nb_series_irm")
)

print("Participants :", visites_irm["PTID"].nunique(), "(attendu : 359)")
print("Séries IRM distinctes :", series["Image ID"].nunique(), "(attendu : 2185)")
print("Visites IRM :", len(visites_irm), "(attendu : 1085)")
print("Visites avec une seule série :", (visites_irm["nb_series_irm"] == 1).sum())
print("Visites avec plusieurs séries :", (visites_irm["nb_series_irm"] > 1).sum())

In [ ]:
# Reprendre les identifiants nettoyés dans la cellule 2.
details = series.copy()
details["description"] = (
    irm["Description"]
    .str.strip()
    .str.upper()
    .replace("", pd.NA)
    .fillna("DESCRIPTION ABSENTE")
)

# Garder uniquement les visites ayant plusieurs séries.
visites_multiples = visites_irm.loc[
    visites_irm["nb_series_irm"] > 1, ["PTID", "date_irm"]
]
details = details.merge(visites_multiples, on=["PTID", "date_irm"],
                        how="inner", validate="many_to_one")

# Regrouper les descriptions de chaque visite, puis compter les combinaisons.
combinaisons = (
    details.groupby(["PTID", "date_irm"])["description"]
    .agg(lambda noms: " + ".join(sorted(noms)))
    .value_counts()
    .rename_axis("descriptions de la visite")
    .reset_index(name="nombre de visites")
)

print("Visites analysées :", combinaisons["nombre de visites"].sum(),
      "(attendu : 1044)")
display(combinaisons.head(10))

In [ ]:
from pathlib import Path
import pandas as pd

# 1. Ouvrir le catalogue IRM déjà téléchargé.
fichier_catalogue = (
    Path(r"E:\Projet_Alzheimer\data\02_Tables_originales")
    / "All_Subjects_Key_MRI_23Oct2026.csv"
)
catalogue = pd.read_csv(fichier_catalogue, dtype="string")

# 2. Rendre comparables les identifiants, par exemple I13722 et 13722.
def normaliser_id(colonne):
    return colonne.str.strip().str.upper().str.extract(r"^I?(\d+)$", expand=False)

candidates = series.copy()
candidates["description_IDA"] = irm["Description"]
candidates["id_commun"] = normaliser_id(candidates["Image ID"])

if candidates["id_commun"].isna().any() or candidates["id_commun"].duplicated().any():
    raise ValueError("Un identifiant de série candidate est absent, invalide ou répété.")

catalogue["id_commun"] = normaliser_id(catalogue["image_id"])
champs = [
    "id_commun", "series_description", "acceleration",
    "acquisition_type", "slice_thickness", "magnetic_field_strength"
]
correspondances = (
    catalogue.loc[catalogue["id_commun"].isin(candidates["id_commun"]), champs]
    .drop_duplicates()
)

# 3. Signaler les identifiants ambigus au lieu de choisir une ligne au hasard.
ambigus = correspondances.loc[
    correspondances["id_commun"].duplicated(keep=False), "id_commun"
].unique()

infos_irm = candidates.merge(
    correspondances.loc[~correspondances["id_commun"].isin(ambigus)],
    on="id_commun", how="left", validate="one_to_one", indicator=True
)

print("Séries candidates :", len(infos_irm))
print("Correspondance unique dans Key MRI :", (infos_irm["_merge"] == "both").sum())
print("Identifiants ambigus dans Key MRI :", len(ambigus))
print("Absentes de Key MRI :", (~candidates["id_commun"].isin(
    correspondances["id_commun"]
)).sum())

In [ ]:
# Une ligne par visite : combien de séries avons-nous,
# et combien sont documentées dans Key MRI ?
couverture_visites = (
    infos_irm.assign(dans_catalogue=infos_irm["_merge"].eq("both"))
    .groupby(["PTID", "date_irm"], as_index=False)
    .agg(
        nb_series=("id_commun", "size"),
        nb_documentees=("dans_catalogue", "sum")
    )
)

couverture_visites["couverture"] = "Une partie des séries"
couverture_visites.loc[
    couverture_visites["nb_documentees"] == 0, "couverture"
] = "Aucune série"
couverture_visites.loc[
    couverture_visites["nb_documentees"] == couverture_visites["nb_series"],
    "couverture"
] = "Toutes les séries"

# Séparer les visites avec une seule série de celles où un choix sera nécessaire.
type_visite = couverture_visites["nb_series"].eq(1).map({
    True: "Une seule série",
    False: "Plusieurs séries"
})

bilan = pd.crosstab(couverture_visites["couverture"], type_visite).reindex(
    index=["Toutes les séries", "Une partie des séries", "Aucune série"],
    columns=["Une seule série", "Plusieurs séries"],
    fill_value=0
)
bilan["Total visites"] = bilan.sum(axis=1)
bilan.loc["Total"] = bilan.sum(axis=0)
display(bilan)

In [ ]:
# 1. Repérer les séries absentes de Key MRI.
# La cellule 4 a confirmé qu'il n'y a aucun identifiant ambigu.
a_completer = infos_irm.loc[
    infos_irm["_merge"].eq("left_only"),
    ["PTID", "date_irm", "id_commun", "description_IDA"]
].merge(
    couverture_visites[["PTID", "date_irm", "nb_series"]],
    on=["PTID", "date_irm"],
    validate="many_to_one"
)

# 2. Ouvrir l'inventaire des en-têtes déjà enregistré dans le dossier cohorte.
fichier_entetes = cohorte / "inventaire_entetes_IRM_20261026_221604_056668.csv"
entetes_irm = pd.read_csv(
    fichier_entetes, sep=None, engine="python", dtype="string"
)
entetes_irm.columns = entetes_irm.columns.str.strip()

print("Séries à compléter :", len(a_completer), "(attendu : 390)")
print("Dont séries de visites avec plusieurs IRM :",
      a_completer["nb_series"].gt(1).sum(), "(attendu : 382)")
print("Lignes dans l’inventaire DICOM :", len(entetes_irm), "(attendu : 2185)")
print("Colonnes de l’inventaire :", " | ".join(entetes_irm.columns))

In [ ]:
# 1. Préparer les identifiants DICOM comme dans la cellule 4.
entetes = entetes_irm.copy()
entetes["id_commun"] = normaliser_id(entetes["Image ID"])

if entetes["id_commun"].isna().any() or entetes["id_commun"].duplicated().any():
    raise ValueError("L’inventaire DICOM contient un identifiant vide ou répété.")

# 2. Relier chaque série à son en-tête, en gardant tout le groupe des 390.
champs = [
    "description", "lignes_pixels", "colonnes_pixels", "espacement_pixels",
    "epaisseur_coupe", "champ_magnetique", "fabricant"
]
complement_dicom = a_completer.merge(
    entetes[["id_commun"] + champs],
    on="id_commun",
    how="left",
    validate="one_to_one",
    indicator="lien_dicom"
)

# 3. Compter les correspondances et les champs renseignés.
retrouvees = complement_dicom["lien_dicom"].eq("both")
print(f"Séries reliées à un en-tête : {retrouvees.sum()} / {len(complement_dicom)}")
print("Séries sans correspondance :", (~retrouvees).sum())

valeurs = complement_dicom[champs].apply(
    lambda colonne: colonne.astype("string").str.strip().replace("", pd.NA)
)
bilan_champs = pd.DataFrame({
    "séries avec une valeur": valeurs.notna().sum(),
    "séries sans valeur": valeurs.isna().sum()
})
display(bilan_champs)

In [ ]:
# Choisir une visite avec deux séries et une couverture Key MRI incomplète.
visites_exemple = couverture_visites.loc[
    couverture_visites["nb_series"].eq(2)
    & couverture_visites["nb_documentees"].lt(2)
].sort_values(["PTID", "date_irm"])

if visites_exemple.empty:
    raise ValueError("Aucune visite avec deux séries ne correspond à ce critère.")

visite = visites_exemple.iloc[0]

# Retrouver les deux acquisitions de cette visite, puis leurs en-têtes.
comparaison = infos_irm.loc[
    infos_irm["PTID"].eq(visite["PTID"])
    & infos_irm["date_irm"].eq(visite["date_irm"]),
    ["id_commun", "description_IDA"]
].merge(
    entetes[["id_commun"] + champs],
    on="id_commun",
    how="left",
    validate="one_to_one"
)

# Une colonne par acquisition : la comparaison se lit ligne par ligne.
print("Nombre de séries pour cette visite :", len(comparaison))
display(comparaison.set_index("id_commun").T)

In [ ]:
# Les six paramètres comparés dans l'exemple.
parametres = [
    "lignes_pixels", "colonnes_pixels", "espacement_pixels",
    "epaisseur_coupe", "champ_magnetique", "fabricant"
]

# Une ligne par série, uniquement pour les visites avec plusieurs IRM.
base = (
    infos_irm[["PTID", "date_irm", "id_commun"]]
    .merge(visites_multiples, on=["PTID", "date_irm"], validate="many_to_one")
    .merge(entetes[["id_commun"] + parametres],
           on="id_commun", how="left", validate="one_to_one")
)

# Comparer les nombres comme des nombres : 1.20 et 1.2 sont équivalents.
numeriques = ["lignes_pixels", "colonnes_pixels",
              "epaisseur_coupe", "champ_magnetique"]
for col in numeriques:
    base[col] = pd.to_numeric(base[col], errors="coerce").round(6)

# Séparer les deux valeurs de l'espacement, par exemple [0.9375, 0.9375].
espacement = base["espacement_pixels"].str.findall(r"\d+(?:\.\d+)?")
for i, col in enumerate(["espacement_1", "espacement_2"]):
    base[col] = pd.to_numeric(
        espacement.where(espacement.str.len().eq(2)).str[i], errors="coerce"
    ).round(6)

base["fabricant"] = base["fabricant"].str.strip().str.upper().replace("", pd.NA)
a_comparer = numeriques + ["espacement_1", "espacement_2", "fabricant"]

# Comparer les séries à l'intérieur de chaque visite.
groupes = base.groupby(["PTID", "date_irm"])[a_comparer]
difference = groupes.nunique().gt(1).any(axis=1)
incomplet = groupes.count().lt(groupes.size(), axis=0).any(axis=1)

resultat_visites = pd.Series(
    "Paramètres concordants", index=difference.index, name="comparaison"
)
resultat_visites.loc[incomplet] = "Comparaison incomplète"
resultat_visites.loc[difference] = "Au moins un paramètre différent"

print("Visites comparées :", len(resultat_visites), "(attendu : 1044)")
print(resultat_visites.value_counts().to_string())

In [ ]:
cles_visite = ["PTID", "date_irm"]
groupes = base.groupby(cles_visite)[a_comparer]

# True si au moins deux séries d'une même visite ont des valeurs différentes.
ecarts = groupes.nunique().gt(1)

# Regrouper les deux dimensions et les deux axes d'espacement pour la lecture.
differences = pd.DataFrame({
    "Dimensions de l’image": ecarts[["lignes_pixels", "colonnes_pixels"]].any(axis=1),
    "Espacement des pixels": ecarts[["espacement_1", "espacement_2"]].any(axis=1),
    "Épaisseur des coupes": ecarts["epaisseur_coupe"],
    "Champ magnétique": ecarts["champ_magnetique"],
    "Fabricant": ecarts["fabricant"],
})

print("Visites avec au moins une différence :",
      int(differences.any(axis=1).sum()), "(attendu : 365)")
display(
    differences.sum().sort_values(ascending=False)
    .rename("nombre de visites concernées").to_frame()
)

# Repérer les champs absents dans la visite classée « Comparaison incomplète ».
visites_incompletes = resultat_visites[
    resultat_visites.eq("Comparaison incomplète")
].index
lignes = base.set_index(cles_visite).loc[visites_incompletes]

print("Valeurs absentes ou non interprétables dans la visite incomplète :")
manques = lignes[a_comparer].isna().sum()
display(manques[manques.gt(0)].rename("nombre de séries concernées").to_frame())

In [ ]:
axes = ["espacement_1", "espacement_2"]

# Comparer les espacements uniquement à l'intérieur d'une même visite.
par_visite = base.groupby(["PTID", "date_irm"])[axes]
ecarts_mm = (par_visite.max() - par_visite.min()).round(6)

# Garder les visites pour lesquelles un écart a été détecté en cellule 10.
visites_avec_ecart = differences.index[differences["Espacement des pixels"]]
ecarts_mm = ecarts_mm.loc[visites_avec_ecart].copy()
ecarts_mm["ecart_max_mm"] = ecarts_mm.max(axis=1)

print("Visites analysées :", len(ecarts_mm), "(attendu : 365)")

# Répartition descriptive : ces tranches ne sont pas des critères d'exclusion.
tranches = pd.cut(
    ecarts_mm["ecart_max_mm"],
    bins=[0, 0.001, 0.01, 0.1, float("inf")],
    labels=["≤ 0,001 mm", "> 0,001 à 0,01 mm",
            "> 0,01 à 0,1 mm", "> 0,1 mm"],
    include_lowest=True
)
display(tranches.value_counts(sort=False).rename("nombre de visites").to_frame())
print("Plus grand écart observé :", ecarts_mm["ecart_max_mm"].max(), "mm")

In [ ]:
# Choisir une visite proche de l'écart médian parmi les 364 cas courants.
courants = ecarts_mm.loc[ecarts_mm["ecart_max_mm"] <= 0.1, "ecart_max_mm"]
visite_typique = (courants - courants.median()).abs().idxmin()

visite_max = ecarts_mm["ecart_max_mm"].idxmax()
visite_incomplete = resultat_visites[
    resultat_visites.eq("Comparaison incomplète")
].index.item()

cas = [
    ("1. Écart typique", visite_typique),
    ("2. Écart maximal", visite_max),
    ("3. Informations incomplètes", visite_incomplete),
]

for nom, (ptid, date) in cas:
    selection = base.loc[
        base["PTID"].eq(ptid) & base["date_irm"].eq(date),
        ["id_commun", "espacement_1", "espacement_2", "epaisseur_coupe"]
    ].merge(
        infos_irm[["id_commun", "description_IDA"]],
        on="id_commun", validate="one_to_one"
    )

    print(f"\n{nom} | {len(selection)} séries")
    display(selection[[
        "id_commun", "description_IDA",
        "espacement_1", "espacement_2", "epaisseur_coupe"
    ]])

In [ ]:
from pathlib import Path
import pydicom
import pandas as pd

ids = ["259688", "259694"]
fichiers = entetes.set_index("id_commun").loc[ids, "fichier_exemple"]
resultats = []

def mesures(objet):
    esp = objet.get("PixelSpacing")
    ep = objet.get("SliceThickness")
    return (
        " × ".join(str(v) for v in esp) if esp else "absent",
        str(ep) if ep is not None else "absent"
    )

for identifiant, chemin in fichiers.items():
    ds = pydicom.dcmread(Path(chemin), stop_before_pixels=True)
    nb_images = int(ds.get("NumberOfFrames", 1))

    # Vérifier d'abord les champs directs.
    observations = [("En-tête principal", *mesures(ds))]

    # Puis vérifier les mesures communes et celles de chaque image.
    for cle, source in [
        ("SharedFunctionalGroupsSequence", "Commun à toutes les images"),
        ("PerFrameFunctionalGroupsSequence", "Par image"),
    ]:
        for groupe in ds.get(cle, []):
            for bloc in groupe.get("PixelMeasuresSequence", []):
                observations.append((source, *mesures(bloc)))

    for source, esp, ep in observations:
        resultats.append([identifiant, nb_images, source, esp, ep])

rapport = pd.DataFrame(resultats, columns=[
    "Série", "Images dans le fichier", "Emplacement",
    "Espacement (mm)", "Épaisseur (mm)"
])
display(rapport.drop_duplicates().reset_index(drop=True))

In [ ]:
from datetime import datetime

# 1. Récupérer les mesures trouvées dans les deux fichiers DICOM.
ids = ["259688", "259694"]
mesures = (
    rapport.loc[
        rapport["Série"].isin(ids) & rapport["Emplacement"].eq("Par image"),
        ["Série", "Espacement (mm)", "Épaisseur (mm)"]
    ]
    .drop_duplicates()
    .set_index("Série")
)

if set(mesures.index) != set(ids) or not mesures.index.is_unique:
    raise ValueError("Mesures absentes ou variables selon les images : vérifier le rapport.")

# 2. Compléter uniquement les deux séries dans une copie du tableau de travail.
base_complete = base.copy()
for identifiant, ligne in mesures.iterrows():
    x, y = map(float, ligne["Espacement (mm)"].split("×"))
    epaisseur = float(ligne["Épaisseur (mm)"])
    masque = base_complete["id_commun"].eq(identifiant)
    if masque.sum() != 1:
        raise ValueError(f"Série {identifiant} absente ou dupliquée.")
    base_complete.loc[masque, ["espacement_1", "espacement_2", "epaisseur_coupe"]] = (
        x, y, epaisseur
    )

# 3. Recalculer le bilan pour les visites ayant plusieurs séries.
groupes = base_complete.groupby(["PTID", "date_irm"])[a_comparer]
different = groupes.nunique().gt(1).any(axis=1)
incomplet = groupes.count().lt(groupes.size(), axis=0).any(axis=1)

bilan_final = pd.Series("Paramètres concordants", index=different.index,
                        name="comparaison")
bilan_final.loc[incomplet] = "Comparaison incomplète"
bilan_final.loc[different] = "Au moins un paramètre différent"

# 4. Garder une trace des mesures récupérées et du résultat par visite.
sortie = cohorte / ("audit_visites_IRM_" + datetime.now().strftime("%Y%m%d_%H%M%S_%f"))
sortie.mkdir()
mesures.to_csv(sortie / "mesures_DICOM_par_image.csv", encoding="utf-8-sig")
bilan_final.to_csv(sortie / "comparaison_visites_multiseries.csv", encoding="utf-8-sig")

print(bilan_final.value_counts().to_string())
print("Visites comparées :", len(bilan_final))
print("Dossier enregistré :", sortie)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pydicom

series_a_voir = {"259688": "MPRAGE SENSE2", "259694": "MPRAGE"}
chemins = entetes.set_index("id_commun")["fichier_exemple"]

fig, axes = plt.subplots(3, 2, figsize=(9, 10), constrained_layout=True)

for col, (identifiant, nom) in enumerate(series_a_voir.items()):
    ds = pydicom.dcmread(chemins.loc[identifiant])
    volume = ds.pixel_array

    if volume.ndim != 3 or volume.shape[0] != 170:
        raise ValueError(f"Série {identifiant} : dimensions inattendues {volume.shape}")

    # Même contraste pour les trois coupes d'une acquisition.
    indices = [round((volume.shape[0] - 1) * f) for f in (0.25, 0.50, 0.75)]
    coupes = volume[indices]
    signal = coupes[coupes != 0]
    if not signal.size:
        raise ValueError(f"Série {identifiant} : les trois coupes sont vides.")
    bas, haut = np.percentile(signal, [1, 99])
    palette = "gray_r" if ds.get("PhotometricInterpretation") == "MONOCHROME1" else "gray"

    for ligne, indice in enumerate(indices):
        ax = axes[ligne, col]
        ax.imshow(volume[indice], cmap=palette, vmin=bas, vmax=haut)
        ax.set_title(f"{nom} · {identifiant}\nCoupe {indice + 1} / 170")
        ax.axis("off")

plt.show()

In [ ]:
import pandas as pd

observation = pd.DataFrame({
    "id_serie": ["259688", "259694"],
    "description": ["MPRAGE SENSE2", "MPRAGE"],
    "controle_effectue": ["3 coupes sur 170"] * 2,
    "observation": ["Pas de flou marqué ni de dédoublement évident sur ces coupes"] * 2,
    "decision": ["Candidate conservée ; contrôle du volume entier à faire"] * 2,
})

fichier = sortie / "observation_visuelle_259688_259694.csv"
observation.to_csv(fichier, index=False, encoding="utf-8-sig")
print("Observation enregistrée :", fichier)

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import pydicom

# 1. Retrouver les acquisitions de la visite choisie en cellule 12.
ptid, date = visite_max
choix = base_complete.loc[
    base_complete["PTID"].eq(ptid) & base_complete["date_irm"].eq(date),
    ["id_commun", "espacement_1", "espacement_2"]
].merge(
    entetes[["id_commun", "description", "fichier_exemple"]],
    on="id_commun", validate="one_to_one"
)

print("Écart maximal entre les séries :",
      round(ecarts_mm.loc[visite_max, "ecart_max_mm"], 4), "mm")
display(choix.drop(columns="fichier_exemple"))

# 2. Trier les fichiers d'une série par position et lire trois coupes.
def trois_coupes(chemin):
    exemple = Path(chemin)
    ref = pydicom.dcmread(exemple, stop_before_pixels=True)
    axes = np.asarray(ref.ImageOrientationPatient, dtype=float)
    normale = np.cross(axes[:3], axes[3:])
    positions = []

    for fichier in exemple.parent.glob("*.dcm"):
        h = pydicom.dcmread(fichier, stop_before_pixels=True)
        if h.SeriesInstanceUID != ref.SeriesInstanceUID:
            continue
        if int(h.get("NumberOfFrames", 1)) != 1:
            raise ValueError("Série multiframe : utiliser une lecture par image.")
        if not np.allclose(h.ImageOrientationPatient, axes, atol=1e-4):
            raise ValueError("Orientation variable : vérifier la géométrie de la série.")
        position = np.dot(np.asarray(h.ImagePositionPatient, dtype=float), normale)
        positions.append((position, fichier))

    positions.sort(key=lambda x: x[0])
    if len(positions) < 3:
        raise ValueError("Moins de trois coupes trouvées dans ce dossier.")
    if np.any(np.diff([p for p, _ in positions]) < 0.001):
        raise ValueError("Positions répétées : vérifier la série avant affichage.")

    rangs = [round((len(positions) - 1) * f) for f in (0.25, 0.50, 0.75)]
    images = [pydicom.dcmread(positions[i][1]).pixel_array for i in rangs]
    palette = "gray_r" if ref.PhotometricInterpretation == "MONOCHROME1" else "gray"
    return images, palette

# 3. Une colonne par acquisition, avec un contraste constant dans la colonne.
fig, axes_plot = plt.subplots(3, len(choix), squeeze=False,
                              figsize=(4 * len(choix), 9), constrained_layout=True)

for col, serie in enumerate(choix.itertuples(index=False)):
    images, palette = trois_coupes(serie.fichier_exemple)
    pixels = np.concatenate([im.ravel() for im in images])
    signal = pixels[pixels != 0]
    if not signal.size:
        raise ValueError(f"Trois coupes vides pour la série {serie.id_commun}.")
    bas, haut = np.percentile(signal, [1, 99])

    for ligne, im in enumerate(images):
        ax = axes_plot[ligne, col]
        ax.imshow(im, cmap=palette, vmin=bas, vmax=haut)
        ax.set_title(f"{serie.description} · {serie.id_commun}"
                     if ligne == 0 else ["", "Milieu", "Trois quarts"][ligne])
        ax.axis("off")

plt.show()

In [ ]:
observation = choix[
    ["id_commun", "description", "espacement_1", "espacement_2"]
].copy()

observation["controle_effectue"] = "Aperçu de coupes sagittales ; revue partielle"
observation["observation"] = (
    "Cadrage différent. Pas de flou marqué ni de dédoublement évident "
    "sur les zones visibles."
)
observation["decision"] = "Candidate conservée ; volume entier à vérifier"

fichier = sortie / "observation_visuelle_190917_190919.csv"
observation.to_csv(fichier, index=False, encoding="utf-8-sig")
print("Observation enregistrée :", fichier)

In [ ]:
from pathlib import Path
import pandas as pd

fichier = (
    Path(r"E:\Projet_Alzheimer\data\02_Tables_originales")
    / "DATADIC_23Oct2026.csv"
)
dico = pd.read_csv(fichier, dtype="string", low_memory=False)

tables_qc = [
    "MRIQUALITY",
    "MAYOADIRL_MRI_IMAGEQC",
    "MAYOADIRL_MRI_QUALITY_ADNI3",
    "MRIQC",
]

dico["table"] = dico["TBLNAME"].str.strip().str.upper()

for table in tables_qc:
    lignes = dico.loc[dico["table"].eq(table)]
    if lignes.empty:
        print(f"\n{table} : non trouvée dans ce dictionnaire")
        continue

    champs = lignes["FLDNAME"].dropna().str.strip().str.upper().unique()
    print(f"\n{table} ({len(champs)} champs)")
    print(" | ".join(sorted(champs)))

In [ ]:
from pathlib import Path
import pandas as pd

dossier_qc = Path(r"E:\Projet_Alzheimer\data\02_Tables_originales\download (2)")
fichier_qc = dossier_qc / "MAYOADIRL_MRI_QUALITY_ADNI3_04Nov2026.csv"

qc_adni3 = pd.read_csv(fichier_qc, dtype="string", low_memory=False)
qc_adni3.columns = qc_adni3.columns.str.strip().str.upper()

print("Fichier lu :", fichier_qc.name)
print("Nombre de lignes :", len(qc_adni3))
print("Colonnes :", " | ".join(qc_adni3.columns))

In [ ]:
# 1. Reprendre les identifiants déjà vérifiés de nos 2 185 séries.
projet = infos_irm[["id_commun", "PTID", "date_irm"]].copy()

# 2. Uniformiser les identifiants Mayo : I12345 et 12345 deviennent 12345.
mayo = qc_adni3.copy()
mayo["id_commun"] = normaliser_id(mayo["LONI_IMAGE"])

# 3. Garder les lignes Mayo dont l'identifiant figure dans notre projet.
mayo_projet = mayo.loc[mayo["id_commun"].isin(projet["id_commun"])].copy()
nb_lignes_par_id = mayo_projet["id_commun"].value_counts()
retrouvees = projet.loc[projet["id_commun"].isin(nb_lignes_par_id.index)]

print(f"Séries retrouvées par LONI_IMAGE : {len(retrouvees)} / {len(projet)}")
print("Séries sans correspondance par cet identifiant :", len(projet) - len(retrouvees))
print("Participants avec au moins une série retrouvée :", retrouvees["PTID"].nunique())
print("Visites avec au moins une série retrouvée :",
      len(retrouvees[["PTID", "date_irm"]].drop_duplicates()))
print("Séries retrouvées sur plusieurs lignes Mayo :", int((nb_lignes_par_id > 1).sum()))

# 4. Afficher les codes bruts sans supposer leur signification.
for champ in ["SERIES_QUALITY", "SERIES_SELECTED"]:
    print(f"\n{champ} — nombre de séries distinctes par valeur :")
    if mayo_projet.empty:
        print("Aucune correspondance par LONI_IMAGE.")
    else:
        valeurs = mayo_projet[champ].str.strip().replace("", pd.NA).fillna("(vide)")
        print(mayo_projet.groupby(valeurs)["id_commun"].nunique().to_string())

In [ ]:
# Une ligne par série du projet, avec son UID lu dans le DICOM.
p = projet.merge(
    entetes[["id_commun", "series_uid"]],
    on="id_commun", how="left", validate="one_to_one"
)
m = mayo.reset_index(drop=True).copy()

def nettoyer_uid(col):
    return col.astype("string").str.strip().replace("", pd.NA)

p["uid"] = nettoyer_uid(p["series_uid"])
m["uid"] = nettoyer_uid(m["SERIESINSTANCEUID"])

# Associer une clé à une ligne Mayo uniquement si cette clé est unique.
def index_unique(col):
    valeurs = m[col].dropna()
    uniques = valeurs[~valeurs.duplicated(keep=False)]
    return pd.Series(uniques.index, index=uniques.values, dtype="Int64")

par_image = p["id_commun"].map(index_unique("id_commun")).astype("Int64")
par_uid = p["uid"].map(index_unique("uid")).astype("Int64")

ambigu = (
    (p["id_commun"].isin(m["id_commun"].dropna()) & par_image.isna())
    | (p["uid"].isin(m["uid"].dropna()) & par_uid.isna())
    | (p["uid"].notna() & p["uid"].duplicated(keep=False))
)
conflit = (par_image.notna() & par_uid.notna() & par_image.ne(par_uid)).fillna(False)

p["ligne_mayo"] = par_image.combine_first(par_uid)
# Une même ligne Mayo ne doit pas être attribuée à deux séries du projet.
reuse = p["ligne_mayo"].notna() & p["ligne_mayo"].duplicated(keep=False)
a_verifier = ambigu | conflit | reuse
p.loc[a_verifier, "ligne_mayo"] = pd.NA

print("Séries reliées sans ambiguïté :", p["ligne_mayo"].notna().sum())
print("Dont nouvelles, retrouvées uniquement par UID :",
      (p["ligne_mayo"].notna() & par_image.isna()).sum())
print("Séries à vérifier (ambiguïté ou conflit) :", a_verifier.sum())
print("Séries sans correspondance par les deux clés :",
      (p["ligne_mayo"].isna() & ~a_verifier).sum())

# Compter les TRUE uniquement parmi les séries reliées sans ambiguïté.
liees = p.loc[p["ligne_mayo"].notna()].copy()
liees["selection"] = liees["ligne_mayo"].map(m["SERIES_SELECTED"]).str.strip().str.upper()
par_visite = (
    liees.assign(est_true=liees["selection"].eq("TRUE").fillna(False))
    .groupby(["PTID", "date_irm"])["est_true"].sum()
)
print("\nNombre de TRUE parmi les séries reliées de chaque visite :")
print(par_visite.value_counts().sort_index().rename_axis("nombre de TRUE")
      .rename("nombre de visites").to_string())

In [ ]:
from pathlib import Path
import pandas as pd

fichier_general = Path(
    r"E:\Projet_Alzheimer\data\02_Tables_originales\QC_Mayo_general"
) / "MRIQC_04Nov2026.csv"

qc_general = pd.read_csv(fichier_general, dtype="string", low_memory=False)
qc_general.columns = qc_general.columns.str.strip().str.upper()

print("Fichier :", fichier_general.name)
print("Nombre de lignes :", len(qc_general))
print("Colonnes :", " | ".join(qc_general.columns))

# Comparer séparément les identifiants possibles, sans les confondre.
ids_projet = set(projet["id_commun"].dropna())
ids_deja_relies = set(liees["id_commun"].dropna())  # 43 séries de la cellule 22

for cle in ["IMAGE_ID", "LONIIMAGE"]:
    if cle not in qc_general.columns:
        print(f"\n{cle} : colonne absente")
        continue

    ids = normaliser_id(qc_general[cle])
    dans_projet = ids.isin(ids_projet)
    retrouves = set(ids[dans_projet])
    nouvelles = retrouves - ids_deja_relies

    print(f"\nRecherche par {cle}")
    print("  Séries du projet retrouvées :", len(retrouves), "/", len(ids_projet))
    print("  Dont absentes des 43 déjà reliées :", len(nouvelles))

    for champ in ["SERIESQC", "SERIESCHOSEN"]:
        if champ not in qc_general.columns:
            print(f"  {champ} : colonne absente")
            continue

        valeurs = qc_general[champ].str.strip().replace("", pd.NA)
        nb = ids[dans_projet & valeurs.notna()].nunique()
        print(f"  {champ} : valeur non vide pour {nb} séries sur {len(retrouves)}")

In [ ]:
from pathlib import Path
import pandas as pd

dossier_rang = Path(r"E:\Projet_Alzheimer\data\02_Tables_originales\QC_ADNI1_Ranking")

# 1. Lire la table de classement ADNI1.
fichier_rang = dossier_rang / "MRIMPRANK_04Nov2026.csv"
rang_adni1 = pd.read_csv(fichier_rang, dtype="string", low_memory=False)
rang_adni1.columns = rang_adni1.columns.str.strip().str.upper()

print("Fichier :", fichier_rang.name)
print("Nombre de lignes :", len(rang_adni1))
print("Colonnes :", " | ".join(rang_adni1.columns))

# 2. Lire les définitions de MRIMPRANK dans le dictionnaire du téléchargement.
fichier_dico = dossier_rang / "DATADIC_04Nov2026.csv"
if fichier_dico.is_file():
    dico_rang = pd.read_csv(fichier_dico, dtype="string", low_memory=False)
    dico_rang.columns = dico_rang.columns.str.strip().str.upper()

    definitions = (
        dico_rang.loc[
            dico_rang["TBLNAME"].str.strip().str.upper().eq("MRIMPRANK"),
            ["FLDNAME", "TEXT", "CODE"]
        ]
        .drop_duplicates()
    )
    if definitions.empty:
        print("Aucune définition trouvée pour MRIMPRANK.")
    else:
        print("\nDéfinitions du dictionnaire :")
        print(definitions.fillna("").to_string(index=False))
else:
    print("Dictionnaire non trouvé dans ce dossier ; la table de classement est chargée.")

In [ ]:
from pathlib import Path
import pandas as pd

fichier = (
    Path(r"E:\Projet_Alzheimer\data\02_Tables_originales\QC_ADNI1_Process")
    / "MRIMPPRO_04Nov2026.csv"
)
process = pd.read_csv(fichier, dtype="string", low_memory=False)
process.columns = process.columns.str.strip().str.upper()
print("Fichier :", fichier.name, "→", len(process), "lignes")
print("Colonnes :", " | ".join(process.columns))

# Identifier la visite et l'identifiant préféré indiqué par MRIMPPRO.
requis = {"PTID", "EXAMDATE", "LONIUID"}
if not requis.issubset(process.columns):
    print("Colonnes à vérifier avant le rapprochement :", requis - set(process.columns))
else:
    pref = pd.DataFrame({
        "PTID": process["PTID"].str.strip(),
        "date_irm": pd.to_datetime(process["EXAMDATE"], errors="coerce").dt.normalize(),
        "uid": process["LONIUID"].str.strip().str.upper()
               .str.extract(r"^[IS]?(\d+)$", expand=False)
    })

    print("Lignes sans participant, date ou identifiant exploitable :",
          pref.isna().any(axis=1).sum())
    pref = pref.dropna().drop_duplicates()

    # Lire les numéros S et I dans les chemins déjà inventoriés.
    # Aucun fichier DICOM n'est ouvert.
    candidats = projet.merge(
        entetes[["id_commun", "fichier_exemple"]],
        on="id_commun", how="left", validate="one_to_one"
    )
    numeros = candidats["fichier_exemple"].str.extract(
        r"(?i)_S(\d+)_I(\d+)\.dcm$"
    )
    candidats["numero_S"] = numeros[0]
    candidats["numero_I"] = candidats["id_commun"]

    print("Numéros S lisibles dans les chemins :", candidats["numero_S"].notna().sum(),
          "/", len(candidats))
    incoherent = numeros[1].notna() & numeros[1].ne(candidats["numero_I"])
    if incoherent.any():
        raise ValueError("Un numéro I du fichier diffère de celui du manifeste.")

    # Tester séparément les deux hypothèses sur les 1 085 visites.
    cles_visite = ["PTID", "date_irm"]
    essais_process = {}
    for cle in ["numero_S", "numero_I"]:
        liens = candidats.dropna(subset=[cle]).merge(
            pref, left_on=cles_visite + [cle], right_on=cles_visite + ["uid"],
            how="inner"
        )
        essais_process[cle] = liens
        par_visite = liens.groupby(cles_visite)["id_commun"].nunique()

        print(f"\nHypothèse LONIUID = {cle}")
        print("  Séries candidates retrouvées :", liens["id_commun"].nunique())
        print("  Visites avec une seule candidate correspondante :", par_visite.eq(1).sum())
        print("  Visites avec plusieurs candidates correspondantes :", par_visite.gt(1).sum())

In [ ]:
# 1. Préparer les notes de MRIMPRANK (table chargée en cellule 24).
r = rang_adni1[["PTID", "EXAMDATE", "LONIUID", "RANK"]].copy()
r["PTID"] = r["PTID"].str.strip()
r["date_irm"] = pd.to_datetime(r["EXAMDATE"], errors="coerce").dt.normalize()
r["uid"] = (
    r["LONIUID"].str.strip().str.upper()
    .str.extract(r"^[IS]?(\d+)$", expand=False)
)
r["note"] = r["RANK"].str.strip().replace("", pd.NA)

cles = ["PTID", "date_irm"]
print("Lignes MRIMPRANK sans clé complète :", r[cles + ["uid"]].isna().any(axis=1).sum())
r = r.dropna(subset=cles + ["uid"])[cles + ["uid", "note"]].drop_duplicates()

# 2. Retrouver les numéros S et I sans ouvrir les DICOM.
c = projet.merge(
    entetes[["id_commun", "fichier_exemple"]],
    on="id_commun", how="left", validate="one_to_one"
)
numeros = c["fichier_exemple"].str.extract(r"(?i)_S(\d+)_I(\d+)\.dcm$")
c["numero_S"] = numeros[0]
c["numero_I"] = c["id_commun"]

if (numeros[1].notna() & numeros[1].ne(c["numero_I"])).any():
    raise ValueError("Un numéro I du fichier diffère de celui du manifeste.")

print("Numéros S lisibles :", c["numero_S"].notna().sum(), "/", len(c))

# 3. Mesurer la couverture de chaque hypothèse, sans choisir de série.
essais_rang = {}
for cle in ["numero_S", "numero_I"]:
    liens = c.dropna(subset=[cle]).merge(
        r, left_on=cles + [cle], right_on=cles + ["uid"], how="inner"
    )
    essais_rang[cle] = liens
    notes_par_serie = liens.groupby("id_commun")["note"].nunique()

    print(f"\nHypothèse LONIUID = {cle}")
    print("  Séries retrouvées :", len(notes_par_serie))
    print("  Visites couvertes :", len(liens[cles].drop_duplicates()))
    print("  Séries avec exactement une note renseignée :", notes_par_serie.eq(1).sum())
    print("  Séries avec des notes contradictoires :", notes_par_serie.gt(1).sum())

In [ ]:
# Une note par série retrouvée avec la clé S.
notes_s = (
    essais_rang["numero_S"][["id_commun", "note"]]
    .dropna(subset=["note"])
    .drop_duplicates()
)
if notes_s["id_commun"].duplicated().any():
    raise ValueError("Plusieurs notes pour une même série : vérifier avant de continuer.")

print("Notes présentes parmi les séries retrouvées :")
print(notes_s["note"].value_counts().to_string())

# Garder les 2 185 séries, même celles que MRIMPRANK ne couvre pas.
audit_rang = projet.merge(notes_s, on="id_commun", how="left", validate="one_to_one")

def bilan_visite(groupe):
    notes = groupe["note"].dropna()
    if notes.empty:
        return "Aucune série notée"
    if len(notes) < len(groupe):
        return "Classement incomplet"
    valeurs = pd.to_numeric(notes, errors="coerce")
    if not valeurs.isin([1, 2, 3]).all():
        return "Code hors 1–3 à vérifier"
    if len(groupe) == 1:
        return "Une seule série candidate, notée"
    if valeurs.eq(valeurs.min()).sum() == 1:
        return "Plusieurs séries : une seule mieux notée"
    return "Plusieurs séries : égalité au meilleur rang"

statut_rang = pd.Series({
    visite: bilan_visite(groupe)
    for visite, groupe in audit_rang.groupby(["PTID", "date_irm"])
}, name="statut_rang")

print("\nBilan des visites :")
print(statut_rang.value_counts().to_string())
print("\nTotal :", len(statut_rang), "(attendu : 1085)")

In [ ]:
# 1. Partir des 1 085 visites, y compris celles sans proposition.
cles = ["PTID", "date_irm"]
suivi = visites_irm[cles + ["nb_series_irm"]].copy()
suivi["statut_rang"] = [
    statut_rang.loc[(ptid, date)] for ptid, date in suivi[cles].itertuples(index=False)
]

# 2. ADNI1 : proposer une série seulement pour les 182 visites
# où toutes les candidates sont notées et une seule a le meilleur rang.
eligibles = suivi.loc[
    suivi["statut_rang"].eq("Plusieurs séries : une seule mieux notée"), cles
]
rangs = audit_rang.merge(eligibles, on=cles, validate="many_to_one")
rangs["rang_num"] = pd.to_numeric(rangs["note"], errors="raise")
meilleur = rangs.groupby(cles)["rang_num"].transform("min")
adni1 = rangs.loc[rangs["rang_num"].eq(meilleur), cles + ["id_commun"]]
adni1 = adni1.rename(columns={"id_commun": "serie_mieux_notee_ADNI1"})

# 3. ADNI3 : reprendre les séries marquées TRUE parmi les liens non ambigus.
adni3 = liees.loc[liees["selection"].eq("TRUE"), cles + ["id_commun"]].copy()
adni3 = adni3.rename(columns={"id_commun": "serie_choisie_Mayo_ADNI3"})

suivi = suivi.merge(adni1, on=cles, how="left", validate="one_to_one")
suivi = suivi.merge(adni3, on=cles, how="left", validate="one_to_one")

a = suivi["serie_mieux_notee_ADNI1"]
b = suivi["serie_choisie_Mayo_ADNI3"]
deux = a.notna() & b.notna()

suivi["source_proposition"] = "Aucune proposition par ces deux règles"
suivi.loc[a.notna() & b.isna(), "source_proposition"] = "Rang ADNI1 seul"
suivi.loc[a.isna() & b.notna(), "source_proposition"] = "Choix Mayo ADNI3 seul"
suivi.loc[deux & a.eq(b), "source_proposition"] = "Deux sources concordantes"
suivi.loc[deux & a.ne(b), "source_proposition"] = "Deux sources en désaccord : à revoir"

fichier_suivi = sortie / "propositions_IRM_par_visite.csv"
suivi.to_csv(fichier_suivi, index=False, encoding="utf-8-sig")

print("Visites :", len(suivi), "(attendu : 1085)")
print("Avec une série mieux notée ADNI1 :", a.notna().sum(), "(attendu : 182)")
print("Avec une série choisie Mayo ADNI3 :", b.notna().sum(), "(attendu : 37)")
print("\nRépartition :")
print(suivi["source_proposition"].value_counts().to_string())
print("\nFichier enregistré :", fichier_suivi)

In [ ]:
# Les 866 visites pour lesquelles ni le rang ADNI1 ni le choix Mayo
# n'ont donné de proposition selon nos règles.
reste = suivi.loc[
    suivi["source_proposition"].eq("Aucune proposition par ces deux règles")
].copy()

reste["action"] = reste["nb_series_irm"].eq(1).map({
    True: "Une seule IRM : vérifier sa qualité",
    False: "Plusieurs IRM : choix à résoudre"
})

print("Visites sans proposition :", len(reste), "(attendu : 866)")
print("\nAction à prévoir :")
print(reste["action"].value_counts().to_string())

# Pour les visites avec plusieurs IRM, préciser pourquoi le rang n'a pas suffi.
a_departager = reste.loc[reste["nb_series_irm"].gt(1)]
print("\nDétail des visites où un choix reste à faire :")
print(a_departager["statut_rang"].value_counts().to_string())

fichier_reste = sortie / "visites_IRM_sans_proposition.csv"
reste.to_csv(fichier_reste, index=False, encoding="utf-8-sig")
print("\nFichier enregistré :", fichier_reste)

In [ ]:
# 1. Reprendre uniquement les visites sans proposition et à égalité.
cles = ["PTID", "date_irm"]
egalites = a_departager.loc[
    a_departager["statut_rang"].eq("Plusieurs séries : égalité au meilleur rang"),
    cles
]
series_egalites = audit_rang.merge(egalites, on=cles, validate="many_to_one")
series_egalites["rang"] = pd.to_numeric(series_egalites["note"], errors="raise")

# 2. Garder toutes les séries qui partagent le meilleur rang de leur visite.
meilleur = series_egalites.groupby(cles)["rang"].transform("min")
a_comparer = series_egalites.loc[
    series_egalites["rang"].eq(meilleur),
    cles + ["id_commun", "rang"]
].copy()

# Ajouter la description pour reconnaître les acquisitions dans IDA.
liste_egalites = a_comparer.merge(
    infos_irm[["id_commun", "description_IDA"]],
    on="id_commun", validate="one_to_one"
).sort_values(cles + ["id_commun"])

par_visite = liste_egalites.groupby(cles).size()
if len(par_visite) != 377 or (par_visite < 2).any():
    raise ValueError("Le résultat ne correspond pas aux 377 visites à égalité.")

fichier = sortie / "IRM_a_egalite_au_meilleur_rang.csv"
liste_egalites.to_csv(fichier, index=False, encoding="utf-8-sig")

print("Visites à égalité :", len(par_visite))
print("Séries à comparer :", len(liste_egalites))
print("Séries de rang moins favorable dans ces visites :",
      len(series_egalites) - len(liste_egalites))
print("\nNombre de visites par nombre de séries à égalité :")
print(par_visite.value_counts().sort_index().to_string())
print("\nFichier enregistré :", fichier)

In [ ]:
cles = ["PTID", "date_irm"]

# 1. Isoler les 462 visites et retrouver toutes leurs séries candidates.
visites_sans_note = a_departager.loc[
    a_departager["statut_rang"].eq("Aucune série notée"), cles
]
series_sans_note = projet.merge(visites_sans_note, on=cles, validate="many_to_one")

# 2. Récupérer les phases renseignées dans MRIQC par Image ID.
phases_qc = pd.DataFrame({
    "id_commun": normaliser_id(qc_general["IMAGE_ID"]),
    "phase": qc_general["MRIPROTOCOLPHASE"].str.strip().str.upper().replace("", pd.NA)
}).dropna().drop_duplicates()

# Si un identifiant a plusieurs phases différentes, ne pas en choisir une.
nb_phases = phases_qc.groupby("id_commun")["phase"].nunique()
phases_uniques = phases_qc[phases_qc["id_commun"].isin(nb_phases[nb_phases.eq(1)].index)]
detail_phases = series_sans_note.merge(
    phases_uniques, on="id_commun", how="left", validate="one_to_one"
)

# 3. Résumer les phases observées pour chaque visite.
def decrire_phase(phases):
    connues = sorted(phases.dropna().unique())
    if not connues:
        return "Non déterminée"
    libelle = " + ".join(connues)
    return libelle + " (couverture partielle)" if phases.isna().any() else libelle

bilan_phases = detail_phases.groupby(cles)["phase"].agg(decrire_phase)

print("Visites analysées :", len(bilan_phases), "(attendu : 462)")
print("Séries candidates dans ces visites :", len(detail_phases))
print("\nVisites par phase de protocole IRM :")
print(bilan_phases.value_counts().to_string())

In [ ]:
from pathlib import Path
import pandas as pd

audit = Path(r"E:\Projet_Alzheimer\data\05_Manifestes") / (
    "cohorte_359_20261024_184212_495491"
) / "audit_visites_IRM_20261104_143324_870635"

visites = pd.read_csv(
    audit / "propositions_IRM_par_visite.csv",
    dtype="string",
    parse_dates=["date_irm"]
)

# Garder uniquement les visites avec une proposition issue du rang ou de Mayo.
avec_proposition = visites.loc[
    visites["source_proposition"].isin(["Rang ADNI1 seul", "Choix Mayo ADNI3 seul"])
].copy()

# Compter les dates distinctes de chaque participant et la durée entre elles.
suivi_propose = (
    avec_proposition.groupby("PTID")["date_irm"]
    .agg(nb_dates="nunique", premiere="min", derniere="max")
)
suivi_propose["duree_jours"] = (
    suivi_propose["derniere"] - suivi_propose["premiere"]
).dt.days

print("Visites avec proposition :", len(avec_proposition), "(attendu : 219)")
print("Participants avec au moins une visite proposée :", len(suivi_propose))
print("Participants avec au moins deux dates proposées :",
      suivi_propose["nb_dates"].ge(2).sum())
print("Dont participants avec au moins 365 jours entre ces dates :",
      (suivi_propose["nb_dates"].ge(2) & suivi_propose["duree_jours"].ge(365)).sum())

In [ ]:
# Choisir le premier identifiant par ordre alphabétique pour que
# la cellule donne le même résultat à chaque exécution.
eligibles = suivi_propose.loc[
    suivi_propose["nb_dates"].ge(2) & suivi_propose["duree_jours"].ge(365)
].sort_index()

ptid = eligibles.index[0]
dates = eligibles.loc[ptid, ["premiere", "derniere"]].tolist()

paire = avec_proposition.loc[
    avec_proposition["PTID"].eq(ptid) & avec_proposition["date_irm"].isin(dates)
].sort_values("date_irm").copy()

paire["id_irm"] = paire["serie_mieux_notee_ADNI1"].fillna(
    paire["serie_choisie_Mayo_ADNI3"]
)
if len(paire) != 2 or paire["id_irm"].nunique() != 2:
    raise ValueError("Deux visites et deux identifiants IRM distincts sont attendus.")

# Retrouver le fichier DICOM exemple déjà enregistré pour chaque série.
paire = paire.merge(
    entetes[["id_commun", "fichier_exemple"]],
    left_on="id_irm", right_on="id_commun", how="left", validate="one_to_one"
)
paire["fichier_accessible"] = paire["fichier_exemple"].map(
    lambda chemin: pd.notna(chemin) and Path(chemin).is_file()
)

print("Intervalle entre les deux IRM :", (dates[1] - dates[0]).days, "jours")
display(paire[["date_irm", "id_irm", "source_proposition", "fichier_accessible"]])

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import pydicom

def verifier_volume(chemin):
    exemple = Path(chemin)
    ref = pydicom.dcmread(exemple, stop_before_pixels=True)
    coupes = []

    for fichier in exemple.parent.glob("*.dcm"):
        h = pydicom.dcmread(fichier, stop_before_pixels=True)
        if h.SeriesInstanceUID == ref.SeriesInstanceUID:
            coupes.append(h)

    if len(coupes) < 3 or any(int(h.get("NumberOfFrames", 1)) != 1 for h in coupes):
        return {"statut": "À revoir : série courte ou multiframe"}

    axes = np.array([h.ImageOrientationPatient for h in coupes], dtype=float)
    positions = np.array([h.ImagePositionPatient for h in coupes], dtype=float)
    pixels = np.array([h.PixelSpacing for h in coupes], dtype=float)

    # Projeter les positions sur l'axe perpendiculaire aux coupes.
    normale = np.cross(axes[0, :3], axes[0, 3:])
    normale /= np.linalg.norm(normale)
    ecarts = np.diff(np.sort(positions @ normale))
    pas = np.median(ecarts)

    alertes = []
    if len({(h.Rows, h.Columns) for h in coupes}) != 1:
        alertes.append("dimensions variables")
    if not np.allclose(axes, axes[0], atol=1e-4, rtol=0):
        alertes.append("orientation variable")
    if not np.allclose(pixels, pixels[0], atol=1e-4, rtol=0):
        alertes.append("espacement des pixels variable")
    if (ecarts < 0.001).any():
        alertes.append("positions répétées ou quasi identiques")
    if not np.allclose(ecarts, pas, atol=0.01, rtol=0.01):
        alertes.append("écart irrégulier entre coupes")

    return {
        "coupes": len(coupes),
        "pas_médian_mm": round(float(pas), 4),
        "écart_min_mm": round(float(ecarts.min()), 4),
        "écart_max_mm": round(float(ecarts.max()), 4),
        "statut": "; ".join(alertes) if alertes else "Géométrie régulière"
    }

resultats = []
for ligne in paire.itertuples(index=False):
    try:
        bilan = verifier_volume(ligne.fichier_exemple)
    except Exception as erreur:
        bilan = {"statut": f"À vérifier : {type(erreur).__name__} : {erreur}"}
    resultats.append({"id_irm": ligne.id_irm, **bilan})

bilan_geometrie = pd.DataFrame(resultats)
display(bilan_geometrie)

In [ ]:
%pip install SimpleITK

In [ ]:
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd
import pydicom
import SimpleITK as sitk

# Un nouveau dossier à chaque exécution : aucun résultat précédent n'est écrasé.
destination = (
    Path(r"E:\Projet_Alzheimer\data")
    / "06_IRM_NIfTI"
    / ("premiere_paire_" + datetime.now().strftime("%Y%m%d_%H%M%S_%f"))
)
destination.mkdir(parents=True, exist_ok=False)

bilan = []
for ligne in paire.itertuples(index=False):
    exemple = Path(ligne.fichier_exemple)
    entete = pydicom.dcmread(exemple, stop_before_pixels=True)
    uid = str(entete.SeriesInstanceUID)

    fichiers = sitk.ImageSeriesReader.GetGDCMSeriesFileNames(str(exemple.parent), uid)
    if len(fichiers) != 166:
        raise ValueError(f"IRM {ligne.id_irm} : {len(fichiers)} fichiers au lieu de 166.")

    lecteur = sitk.ImageSeriesReader()
    lecteur.SetFileNames(fichiers)
    volume = lecteur.Execute()

    if volume.GetDimension() != 3 or volume.GetSize()[2] != 166:
        raise ValueError(f"IRM {ligne.id_irm} : dimensions inattendues {volume.GetSize()}.")

    fichier = destination / f"IRM_{ligne.id_irm}.nii.gz"
    sitk.WriteImage(volume, str(fichier))
    relu = sitk.ReadImage(str(fichier))

    geometrie_ok = (
        relu.GetSize() == volume.GetSize()
        and all(np.allclose(a, b, atol=1e-4, rtol=0) for a, b in [
            (relu.GetSpacing(), volume.GetSpacing()),
            (relu.GetOrigin(), volume.GetOrigin()),
            (relu.GetDirection(), volume.GetDirection()),
        ])
    )
    pixels_ok = np.array_equal(
        sitk.GetArrayViewFromImage(volume), sitk.GetArrayViewFromImage(relu)
    )
    if not (geometrie_ok and pixels_ok):
        raise ValueError(f"IRM {ligne.id_irm} : la vérification du NIfTI a échoué.")

    bilan.append({
        "IRM": ligne.id_irm,
        "dimensions": relu.GetSize(),
        "espacement_mm": tuple(round(x, 4) for x in relu.GetSpacing()),
        "fichier": fichier.name,
        "vérification": "OK",
    })

bilan_conversion = pd.DataFrame(bilan)
bilan_conversion.to_csv(destination / "bilan_conversion.csv", index=False)
display(bilan_conversion)
print("Dossier enregistré :", destination)

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import SimpleITK as sitk

dossier = Path(r"E:\Projet_Alzheimer\data\06_IRM_NIfTI\premiere_paire_20261104_171451_793123")
examens = [("11606", "22/02/2006"), ("103934", "24/04/2008")]

fig, axes = plt.subplots(2, 3, figsize=(12, 8), constrained_layout=True)

for ligne, (identifiant, date) in enumerate(examens):
    image = sitk.ReadImage(str(dossier / f"IRM_{identifiant}.nii.gz"))
    image = sitk.DICOMOrient(image, "LPS")
    volume = sitk.GetArrayFromImage(image)  # ordre NumPy : z, y, x
    nz, ny, nx = volume.shape
    sx, sy, sz = image.GetSpacing()

    # Une coupe centrale par direction ; proportions corrigées en millimètres.
    vues = [
        ("Profil",      volume[:, :, nx // 2], sz / sy, "lower"),
        ("Face",        volume[:, ny // 2, :], sz / sx, "lower"),
        ("Horizontale", volume[nz // 2, :, :], sy / sx, "upper"),
    ]

    signal = volume[np.isfinite(volume) & (volume > 0)]
    if signal.size == 0:
        raise ValueError(f"Aucun signal positif dans l’IRM {identifiant}.")
    bas, haut = np.percentile(signal, [1, 99])

    for col, (nom, coupe, aspect, origine) in enumerate(vues):
        ax = axes[ligne, col]
        ax.imshow(coupe, cmap="gray", origin=origine, aspect=aspect,
                  vmin=bas, vmax=haut, interpolation="nearest")
        ax.set_title(f"{date} · IRM {identifiant}\n{nom}")
        ax.axis("off")

plt.show()

In [ ]:
import pandas as pd

controle_paire = pd.DataFrame({
    "id_irm": ["11606", "103934"],
    "date_irm": ["2006-02-22", "2008-04-24"],
    "controle": "Trois coupes centrales : profil, face, horizontale",
    "observation": [
        "Pas de coupure évidente du cerveau sur les vues centrales.",
        "Pas de coupure évidente sur les vues centrales ; profil plus sombre."
    ],
    "decision": "Candidate conservée",
    "couverture_volume_entier": "À vérifier"
})

fichier = dossier / "controle_visuel_premiere_paire.csv"
controle_paire.to_csv(fichier, index=False, encoding="utf-8-sig")
print("Contrôle enregistré :", fichier)

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import SimpleITK as sitk

dossier = Path(r"E:\Projet_Alzheimer\data\06_IRM_NIfTI\premiere_paire_20261104_171451_793123")

# Charger les deux volumes une seule fois pour que le défilement soit rapide.
vues = {}
for identifiant in ["11606", "103934"]:
    image = sitk.ReadImage(str(dossier / f"IRM_{identifiant}.nii.gz"))
    image = sitk.DICOMOrient(image, "LPS")
    volume = sitk.GetArrayFromImage(image)
    sx, sy, _ = image.GetSpacing()
    signal = volume[np.isfinite(volume) & (volume > 0)]
    bas, haut = np.percentile(signal, [1, 99])
    vues[identifiant] = (volume, sy / sx, bas, haut)

def afficher(identifiant, coupe):
    volume, aspect, bas, haut = vues[identifiant]
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.imshow(volume[coupe - 1], cmap="gray", origin="upper",
              aspect=aspect, vmin=bas, vmax=haut)
    ax.set_title(f"IRM {identifiant} · coupe {coupe} / {len(volume)}")
    ax.axis("off")
    plt.show()
    plt.close(fig)

try:
    import ipywidgets as widgets
    from IPython.display import display

    for identifiant in vues:
        n = len(vues[identifiant][0])
        curseur = widgets.IntSlider(
            value=n // 2, min=1, max=n, step=1,
            description="Coupe", continuous_update=False
        )
        sortie_vue = widgets.interactive_output(
            afficher, {"identifiant": widgets.fixed(identifiant), "coupe": curseur}
        )
        display(widgets.VBox([curseur, sortie_vue]))

except ImportError:
    print("Curseurs indisponibles. Utilise afficher('11606', 100) dans une cellule.")

In [ ]:
fractions = [0.50, 0.70, 0.80, 0.90, 0.95, 1.00]
fig, axes = plt.subplots(2, len(fractions), figsize=(15, 6), constrained_layout=True)

for ligne, identifiant in enumerate(["11606", "103934"]):
    volume, aspect, bas, haut = vues[identifiant]

    for colonne, fraction in enumerate(fractions):
        i = round((len(volume) - 1) * fraction)
        ax = axes[ligne, colonne]
        ax.imshow(volume[i], cmap="gray", origin="upper",
                  aspect=aspect, vmin=bas, vmax=haut)
        ax.set_title(f"IRM {identifiant}\nCoupe {i + 1}/{len(volume)}")
        ax.axis("off")

plt.show()

In [ ]:
fractions = [0.00, 0.10, 0.20, 0.30, 0.40, 0.50]
fig, axes = plt.subplots(2, 6, figsize=(15, 6), constrained_layout=True)

for ligne, identifiant in enumerate(["11606", "103934"]):
    volume, aspect, bas, haut = vues[identifiant]

    for colonne, fraction in enumerate(fractions):
        i = round((len(volume) - 1) * fraction)
        ax = axes[ligne, colonne]
        ax.imshow(volume[i], cmap="gray", origin="upper",
                  aspect=aspect, vmin=bas, vmax=haut)
        ax.set_title(f"IRM {identifiant}\nCoupe {i + 1}/{len(volume)}")
        ax.axis("off")

plt.show()

In [ ]:
from pathlib import Path
import pandas as pd

# Les deux visites pour lesquelles on a déjà un volume IRM converti.
paire_irm = paire[["PTID", "date_irm", "id_irm"]].copy()
paire_irm["date_irm"] = pd.to_datetime(paire_irm["date_irm"]).dt.normalize()
paire_irm["fichier_nifti"] = paire_irm["id_irm"].map(
    lambda identifiant: str(dossier / f"IRM_{identifiant}.nii.gz")
)

# Relire les associations utilisées pour sélectionner la cohorte.
fichier_visites = cohorte / "visites_candidates.csv"
visites_candidates = pd.read_csv(fichier_visites, dtype="string")
visites_candidates.columns = visites_candidates.columns.str.strip()

print("Colonnes du manifeste :", " | ".join(visites_candidates.columns))

cles = ["PTID", "date_irm"]
if not set(cles).issubset(visites_candidates.columns):
    print("Les clés ont un autre nom dans ce manifeste. Voir les colonnes ci-dessus.")
else:
    visites_candidates["PTID"] = visites_candidates["PTID"].str.strip()
    visites_candidates["date_irm"] = pd.to_datetime(
        visites_candidates["date_irm"], errors="raise"
    ).dt.normalize()

    exemple_visites = paire_irm.merge(
        visites_candidates, on=cles, how="left",
        validate="one_to_many", indicator="lien_manifeste"
    )

    # Afficher les informations utiles sans le long chemin du fichier.
    colonnes_utiles = [
        c for c in exemple_visites.columns
        if c in ["id_irm", "date_irm", "lien_manifeste"]
        or any(mot in c.lower() for mot in ["pet", "diag", "dx", "ecart"])
    ]
    display(exemple_visites[colonnes_utiles])

In [ ]:
from pathlib import Path
import pandas as pd

rapport_pet = (
    cohorte / "collecte_PET_20261026_221901_669799"
    / "controle_entetes_PET_20261027_151028_758875.csv"
)
pet = pd.read_csv(rapport_pet, dtype="string")

# Garder les séries PET dont le participant et la date ont été vérifiés.
ok = (
    pet["statut"].eq("OK")
    & pet["modalite"].eq("PT")
    & pet["participant_concordant"].str.lower().eq("true")
    & pet["date_concordante"].str.lower().eq("true")
)
pet = pet.loc[ok, ["PTID", "date_pet", "image_id", "fichier_exemple"]].copy()
pet["PTID"] = pet["PTID"].str.strip()
pet["date_pet"] = pd.to_datetime(pet["date_pet"], errors="raise").dt.normalize()
pet = pet.rename(columns={"image_id": "id_pet", "fichier_exemple": "exemple_dicom_pet"})

# Chercher uniquement les deux visites de la cellule 42.
cles = ["PTID", "date_pet"]
deux_visites = exemple_visites.copy()
deux_visites["date_pet"] = pd.to_datetime(deux_visites["date_pet"]).dt.normalize()
pet_paire = pet.merge(deux_visites[cles].drop_duplicates(), on=cles, how="inner")

if pet_paire.duplicated(cles).any():
    raise ValueError("Plusieurs séries PET pour une même visite : vérifier avant de choisir.")

paire_multimodale = deux_visites.merge(
    pet_paire, on=cles, how="left", validate="many_to_one"
)
paire_multimodale["dicom_pet_accessible"] = paire_multimodale["exemple_dicom_pet"].map(
    lambda f: pd.notna(f) and Path(f).is_file()
)

display(paire_multimodale[
    ["id_irm", "date_irm", "date_pet", "id_pet", "dicom_pet_accessible", "DIAGNOSIS"]
])

In [ ]:
from pathlib import Path
import pandas as pd

fichier_mmse = Path(
    r"E:\Projet_Alzheimer\data\02_Tables_originales"
    r"\All_Subjects_MMSE_23Oct2026.csv"
)
mmse = pd.read_csv(
    fichier_mmse, dtype="string", usecols=["PTID", "VISDATE", "MMSCORE"]
)
mmse["PTID"] = mmse["PTID"].str.strip()
mmse["date_mmse"] = pd.to_datetime(mmse["VISDATE"], errors="coerce").dt.normalize()
mmse["score"] = pd.to_numeric(mmse["MMSCORE"], errors="coerce")

table_paire = paire_multimodale.copy()
table_paire["date_dx"] = pd.to_datetime(table_paire["date_dx"]).dt.normalize()

def trouver_mmse(ptid, date_dx):
    lignes = mmse.loc[mmse["PTID"].eq(ptid) & mmse["date_mmse"].eq(date_dx)]
    if lignes.empty:
        return pd.NA, "Pas de test à cette date"

    scores = lignes.loc[lignes["score"].between(0, 30), "score"].unique()
    if len(scores) == 1:
        return int(scores[0]), "Score retrouvé"
    if len(scores) == 0:
        return pd.NA, "Test présent, sans score valide"
    return pd.NA, "Scores contradictoires à vérifier"

resultats = [
    trouver_mmse(ptid, date)
    for ptid, date in zip(table_paire["PTID"], table_paire["date_dx"])
]
table_paire["MMSE"] = pd.array([r[0] for r in resultats], dtype="Int64")
table_paire["statut_mmse"] = [r[1] for r in resultats]

table_paire.to_csv(dossier / "premiere_paire_IRM_PET_clinique.csv",
                   index=False, encoding="utf-8-sig")

display(table_paire[["id_irm", "id_pet", "date_dx", "DIAGNOSIS", "MMSE", "statut_mmse"]])

In [ ]:
# Relire la table avec les codes de visite pour comprendre le décalage.
tests = pd.read_csv(
    fichier_mmse, dtype="string",
    usecols=["PTID", "PHASE", "VISCODE", "VISCODE2", "VISDATE", "MMSCORE"]
)
ptid_paire = table_paire["PTID"].iloc[0]
tests = tests.loc[tests["PTID"].str.strip().eq(ptid_paire)].copy()
tests["date_mmse"] = pd.to_datetime(tests["VISDATE"], errors="coerce").dt.normalize()

print("Lignes MMSE pour ce participant :", len(tests))
print("Dont lignes sans date exploitable :", tests["date_mmse"].isna().sum())

for visite in table_paire.itertuples(index=False):
    print(f"\nIRM {visite.id_irm} | diagnostic du {visite.date_dx:%d/%m/%Y}")

    candidats = tests.dropna(subset=["date_mmse"]).copy()
    if candidats.empty:
        print("Aucun MMSE daté pour ce participant.")
        continue

    candidats["jours_depuis_dx"] = (candidats["date_mmse"] - visite.date_dx).dt.days
    candidats["jours_depuis_irm"] = (
        candidats["date_mmse"] - pd.Timestamp(visite.date_irm)
    ).dt.days

    # Garder toutes les lignes des trois dates les plus proches, même en cas d'égalité.
    distances = candidats.groupby("date_mmse")["jours_depuis_dx"].first().abs()
    dates_proches = distances.nsmallest(3, keep="all").index
    display(
        candidats.loc[candidats["date_mmse"].isin(dates_proches), [
            "date_mmse", "PHASE", "VISCODE", "VISCODE2", "MMSCORE",
            "jours_depuis_dx", "jours_depuis_irm"
        ]].sort_values("date_mmse")
    )

In [ ]:
from pathlib import Path
import pandas as pd

fichier_dx = Path(
    r"E:\Projet_Alzheimer\data\02_Tables_originales"
    r"\All_Subjects_DXSUM_23Oct2026.csv"
)
dx = pd.read_csv(
    fichier_dx, dtype="string",
    usecols=["PTID", "PHASE", "VISCODE", "VISCODE2", "EXAMDATE", "DIAGNOSIS"]
)
dx["date_dx"] = pd.to_datetime(dx["EXAMDATE"], errors="coerce").dt.normalize()

# Afficher les codes source des diagnostics des deux visites.
dx_paire = dx.loc[
    dx["PTID"].str.strip().eq(ptid_paire)
    & dx["date_dx"].isin(table_paire["date_dx"]),
    ["date_dx", "PHASE", "VISCODE", "VISCODE2", "DIAGNOSIS"]
].drop_duplicates().sort_values("date_dx")

display(dx_paire)

In [ ]:
# Repartir du tableau des deux IRM, sans modifier le fichier précédent.
paire_clinique = table_paire.rename(columns={
    "MMSE": "MMSE_jour_dx",
    "statut_mmse": "statut_mmse_jour_dx"
}).merge(
    dx_paire[["date_dx", "PHASE", "VISCODE2"]].drop_duplicates(),
    on="date_dx", how="left", validate="one_to_one"
)

# 'tests' contient déjà uniquement les MMSE du participant (cellule 45).
def mmse_de_visite(ligne):
    memes_visites = tests.loc[
        tests["PHASE"].str.strip().eq(ligne["PHASE"])
        & tests["VISCODE2"].str.strip().eq(ligne["VISCODE2"])
    ]
    if memes_visites.empty:
        return pd.NA, pd.NaT, "Aucun test avec ce code de visite"

    scores = pd.to_numeric(memes_visites["MMSCORE"], errors="coerce")
    valides = memes_visites.loc[
        scores.between(0, 30) & memes_visites["date_mmse"].notna(), ["date_mmse"]
    ].assign(score=scores).drop_duplicates()

    if len(valides) == 0:
        return pd.NA, pd.NaT, "Aucun score valide et daté"
    if len(valides) > 1:
        return pd.NA, pd.NaT, "Plusieurs dates ou scores : à vérifier"

    return int(valides.iloc[0]["score"]), valides.iloc[0]["date_mmse"], "Associé par visite"

resultats = [mmse_de_visite(ligne) for _, ligne in paire_clinique.iterrows()]
paire_clinique["MMSE_visite"] = pd.array([r[0] for r in resultats], dtype="Int64")
paire_clinique["date_mmse"] = pd.to_datetime([r[1] for r in resultats])
paire_clinique["statut_mmse_visite"] = [r[2] for r in resultats]
paire_clinique["jours_mmse_depuis_irm"] = (
    paire_clinique["date_mmse"] - paire_clinique["date_irm"]
).dt.days.astype("Int64")

paire_clinique.to_csv(dossier / "premiere_paire_clinique_par_visite.csv",
                      index=False, encoding="utf-8-sig")
display(paire_clinique[[
    "id_irm", "VISCODE2", "MMSE_jour_dx", "MMSE_visite",
    "date_mmse", "jours_mmse_depuis_irm", "statut_mmse_visite"
]])

In [ ]:
resultats_pet = []

for visite in paire_clinique.itertuples(index=False):
    try:
        bilan = verifier_volume(visite.exemple_dicom_pet)
    except Exception as erreur:
        bilan = {"statut": f"À vérifier : {type(erreur).__name__} : {erreur}"}

    resultats_pet.append({
        "id_irm": visite.id_irm,
        "id_pet": visite.id_pet,
        "date_pet": visite.date_pet,
        **bilan
    })

geometrie_pet = pd.DataFrame(resultats_pet)
display(geometrie_pet)

In [ ]:
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd
import pydicom
import SimpleITK as sitk

destination_pet = (
    Path(r"E:\Projet_Alzheimer\data\07_PET_NIfTI")
    / ("premiere_paire_" + datetime.now().strftime("%Y%m%d_%H%M%S_%f"))
)
destination_pet.mkdir(parents=True, exist_ok=False)

resultats = []
for visite in paire_clinique.itertuples(index=False):
    exemple = Path(visite.exemple_dicom_pet)
    uid = str(pydicom.dcmread(exemple, stop_before_pixels=True).SeriesInstanceUID)

    # Récupérer uniquement les coupes de ce PET, dans l'ordre spatial.
    fichiers = sitk.ImageSeriesReader.GetGDCMSeriesFileNames(str(exemple.parent), uid)
    if len(fichiers) != 96:
        raise ValueError(f"PET {visite.id_pet} : {len(fichiers)} fichiers au lieu de 96.")

    lecteur = sitk.ImageSeriesReader()
    lecteur.SetFileNames(fichiers)
    lecteur.SetOutputPixelType(sitk.sitkFloat64)
    volume = lecteur.Execute()

    if volume.GetDimension() != 3 or volume.GetSize()[2] != 96:
        raise ValueError(f"PET {visite.id_pet} : dimensions inattendues {volume.GetSize()}.")

    chemin = destination_pet / f"PET_{visite.id_pet}.nii.gz"
    sitk.WriteImage(volume, str(chemin))
    relu = sitk.ReadImage(str(chemin))

    geometrie_ok = (
        volume.GetSize() == relu.GetSize()
        and all(np.allclose(a, b, atol=1e-4, rtol=0) for a, b in [
            (volume.GetSpacing(), relu.GetSpacing()),
            (volume.GetOrigin(), relu.GetOrigin()),
            (volume.GetDirection(), relu.GetDirection())
        ])
    )
    valeurs_ok = np.array_equal(
        sitk.GetArrayViewFromImage(volume), sitk.GetArrayViewFromImage(relu)
    )
    if not (geometrie_ok and valeurs_ok):
        raise ValueError(f"PET {visite.id_pet} : la vérification du NIfTI a échoué.")

    resultats.append({
        "id_pet": visite.id_pet,
        "dimensions": relu.GetSize(),
        "espacement_mm": tuple(round(x, 4) for x in relu.GetSpacing()),
        "fichier_nifti_pet": str(chemin),
        "verification": "OK"
    })

# Enregistrer le bilan et une nouvelle copie du tableau avec les chemins PET.
bilan_pet = pd.DataFrame(resultats)
bilan_pet.to_csv(destination_pet / "bilan_conversion_PET.csv", index=False)

paire_avec_pet = paire_clinique.merge(
    bilan_pet[["id_pet", "fichier_nifti_pet"]],
    on="id_pet", validate="one_to_one"
)
paire_avec_pet.to_csv(destination_pet / "premiere_paire_IRM_PET_clinique.csv", index=False)

display(bilan_pet[["id_pet", "dimensions", "espacement_mm", "verification"]])
print("Dossier enregistré :", destination_pet)

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import SimpleITK as sitk

dossier_pet = Path(
    r"E:\Projet_Alzheimer\data\07_PET_NIfTI"
    r"\premiere_paire_20261104_180032_856043"
)
examens_pet = [("1593815", "08/03/2006"), ("1593829", "24/04/2008")]

fig, axes = plt.subplots(2, 3, figsize=(11, 7), constrained_layout=True)

for ligne, (identifiant, date) in enumerate(examens_pet):
    image = sitk.ReadImage(str(dossier_pet / f"PET_{identifiant}.nii.gz"))
    image = sitk.DICOMOrient(image, "LPS")
    volume = sitk.GetArrayFromImage(image)
    nz, ny, nx = volume.shape
    sx, sy, sz = image.GetSpacing()

    signal = volume[np.isfinite(volume) & (volume > 0)]
    if signal.size == 0:
        raise ValueError(f"PET {identifiant} : aucun signal positif à afficher.")
    haut = np.percentile(signal, 99)

    vues_pet = [
        ("Profil",      volume[:, :, nx // 2], sz / sy, "lower"),
        ("Face",        volume[:, ny // 2, :], sz / sx, "lower"),
        ("Horizontale", volume[nz // 2, :, :], sy / sx, "upper"),
    ]
    for col, (nom, coupe, aspect, origine) in enumerate(vues_pet):
        ax = axes[ligne, col]
        ax.imshow(coupe, cmap="gray", origin=origine, aspect=aspect,
                  vmin=0, vmax=haut, interpolation="nearest")
        ax.set_title(f"PET {identifiant} · {date}\n{nom}")
        ax.axis("off")

plt.show()

In [ ]:
from itertools import combinations
from pathlib import Path
import pandas as pd

# 1. Reprendre les associations IRM–PET–diagnostic déjà établies.
cles_irm = ["PTID", "date_irm"]
associations = visites_candidates[
    ["PTID", "date_irm", "date_pet", "date_dx", "DIAGNOSIS"]
].copy()

for col in ["date_irm", "date_pet", "date_dx"]:
    associations[col] = pd.to_datetime(associations[col], errors="raise").dt.normalize()

liens = avec_proposition.merge(
    associations.drop_duplicates(), on=cles_irm, how="left", validate="one_to_many"
)

# 2. Garder les PET vérifiés en cellule 43 dont le fichier existe toujours.
pet_disque = pet.loc[
    pet["exemple_dicom_pet"].map(lambda f: pd.notna(f) and Path(f).is_file()),
    ["PTID", "date_pet", "id_pet"]
].drop_duplicates()

liens = liens.merge(pet_disque, on=["PTID", "date_pet"], how="inner")
liens = liens.dropna(subset=["date_irm", "date_pet", "date_dx", "DIAGNOSIS", "id_pet"])

# Une ligne par association distincte ; ne pas choisir arbitrairement
# si une même date IRM a plusieurs associations candidates.
liens = liens.drop_duplicates(["PTID", "date_irm", "date_pet", "id_pet", "date_dx"])

# 3. Compter chaque participant une seule fois s'il a au moins une paire valide.
eligibles_multimodaux = []
for ptid, groupe in liens.groupby("PTID"):
    visites_pt = list(groupe.sort_values("date_irm").itertuples(index=False))
    for a, b in combinations(visites_pt, 2):
        if (
            (b.date_irm - a.date_irm).days >= 365
            and a.date_pet != b.date_pet
            and a.id_pet != b.id_pet
            and a.date_dx != b.date_dx
        ):
            eligibles_multimodaux.append(ptid)
            break

print("Visites avec proposition IRM au départ :", len(avec_proposition))
print("Visites distinctes avec PET accessible et diagnostic :",
      len(liens[cles_irm].drop_duplicates()))
print("Participants avec au moins une telle visite :", liens["PTID"].nunique())
print("Participants avec deux visites compatibles à ≥ 365 jours :",
      len(eligibles_multimodaux))

In [ ]:
from itertools import combinations
from pathlib import Path
import pandas as pd

# Reprendre les associations vérifiées dans la cellule 51.
base_paires = liens.loc[liens["PTID"].isin(eligibles_multimodaux)].copy()
base_paires["id_irm"] = base_paires["serie_mieux_notee_ADNI1"].fillna(
    base_paires["serie_choisie_Mayo_ADNI3"]
)

champs = ["date_irm", "id_irm", "source_proposition",
          "date_pet", "id_pet", "date_dx", "DIAGNOSIS"]
base_paires = base_paires[["PTID"] + champs].drop_duplicates()

lignes = []
for ptid, groupe in base_paires.groupby("PTID"):
    visites_pt = groupe.sort_values(["date_irm", "date_pet"]).to_dict("records")

    for debut, fin in combinations(visites_pt, 2):
        jours = (fin["date_irm"] - debut["date_irm"]).days
        if (jours < 365
            or debut["date_pet"] == fin["date_pet"]
            or debut["id_pet"] == fin["id_pet"]
            or debut["date_dx"] == fin["date_dx"]):
            continue

        ligne = {"PTID": ptid, "suivi_jours": jours}
        for champ in champs:
            ligne[f"{champ}_debut"] = debut[champ]
            ligne[f"{champ}_fin"] = fin[champ]
        lignes.append(ligne)

paires_35 = pd.DataFrame(lignes)
if paires_35.empty or set(paires_35["PTID"]) != set(eligibles_multimodaux):
    raise ValueError("Les participants retrouvés diffèrent du bilan de la cellule 51.")

fichier = Path(audit) / "paires_multimodales_compatibles.csv"
paires_35.to_csv(fichier, index=False, encoding="utf-8-sig")

print("Participants enregistrés :", paires_35["PTID"].nunique(), "(attendu : 35)")
print("Paires compatibles :", len(paires_35))
print("Suivi minimal / maximal (jours) :",
      paires_35["suivi_jours"].min(), "/", paires_35["suivi_jours"].max())
print("Fichier enregistré :", fichier)

In [ ]:
import pandas as pd

# 1. Relier chaque diagnostic à sa phase et à son code de visite.
dx_source = dx[["PTID", "date_dx", "PHASE", "VISCODE2"]].copy()
for col in ["PTID", "PHASE", "VISCODE2"]:
    dx_source[col] = dx_source[col].str.strip().replace("", pd.NA)

dx_source = dx_source.dropna().drop_duplicates()
cles_dx = ["PTID", "date_dx"]

# Ne pas choisir arbitrairement si une date correspond à plusieurs visites.
dx_unique = dx_source.loc[~dx_source.duplicated(cles_dx, keep=False)]

# 2. Lire tous les MMSE, pas seulement ceux du premier participant.
m = pd.read_csv(
    fichier_mmse, dtype="string",
    usecols=["PTID", "PHASE", "VISCODE2", "VISDATE", "MMSCORE"]
)
cles_visite = ["PTID", "PHASE", "VISCODE2"]
for col in cles_visite:
    m[col] = m[col].str.strip().replace("", pd.NA)

m["date_mmse"] = pd.to_datetime(m["VISDATE"], errors="coerce").dt.normalize()
m["mmse"] = pd.to_numeric(m["MMSCORE"], errors="coerce")

# Garder une seule combinaison valide (date, score) par visite.
m = m.loc[m["mmse"].between(0, 30), cles_visite + ["date_mmse", "mmse"]]
m = m.dropna().drop_duplicates()
m_unique = m.loc[~m.duplicated(cles_visite, keep=False)]

scores = dx_unique.merge(m_unique, on=cles_visite, how="left", validate="many_to_one")

# 3. Ajouter les scores aux 50 paires sans perdre les paires incomplètes.
audit_mmse = paires_35.copy()
for cote in ["debut", "fin"]:
    colonne_date = f"date_dx_{cote}"
    audit_mmse[colonne_date] = pd.to_datetime(audit_mmse[colonne_date]).dt.normalize()
    source = scores[["PTID", "date_dx", "date_mmse", "mmse"]].rename(columns={
        "date_dx": colonne_date,
        "date_mmse": f"date_mmse_{cote}",
        "mmse": f"mmse_{cote}"
    })
    audit_mmse = audit_mmse.merge(
        source, on=["PTID", colonne_date], how="left", validate="many_to_one"
    )

debut = audit_mmse["mmse_debut"].notna()
fin = audit_mmse["mmse_fin"].notna()

for nom, masque in [
    ("MMSE à la visite de fin", fin),
    ("MMSE aux deux visites", debut & fin),
]:
    print(f"{nom} : {masque.sum()} / {len(audit_mmse)} paires ; "
          f"{audit_mmse.loc[masque, 'PTID'].nunique()} / 35 participants")

fichier = audit / "paires_compatibles_audit_MMSE.csv"
audit_mmse.to_csv(fichier, index=False, encoding="utf-8-sig")
print("Fichier enregistré :", fichier)

In [ ]:
import pandas as pd

horizons = audit_mmse.copy()
dates_depart = ["date_irm_debut", "date_pet_debut", "date_dx_debut"]

for col in dates_depart + ["date_mmse_fin"]:
    horizons[col] = pd.to_datetime(horizons[col], errors="raise").dt.normalize()

if horizons[dates_depart + ["date_mmse_fin"]].isna().any().any():
    raise ValueError("Une date nécessaire au calcul du délai est manquante.")

horizons["date_prediction"] = horizons[dates_depart].max(axis=1)
horizons["delai_mmse_jours"] = (
    horizons["date_mmse_fin"] - horizons["date_prediction"]
).dt.days

# Fenêtres exploratoires, pas encore des critères définitifs du projet.
fenetres = [
    ("Autour de 1 an (275–455 jours)", 275, 455),
    ("Autour de 2 ans (640–820 jours)", 640, 820),
    ("Autour de 3 ans (1 005–1 185 jours)", 1005, 1185),
]

bilan = []
for nom, minimum, maximum in fenetres:
    retenues = horizons[horizons["delai_mmse_jours"].between(minimum, maximum)]
    bilan.append({
        "horizon testé": nom,
        "paires": len(retenues),
        "participants distincts": retenues["PTID"].nunique()
    })

print("Paires dont le MMSE n'est pas postérieur au départ :",
      int(horizons["delai_mmse_jours"].le(0).sum()))
display(pd.DataFrame(bilan))

horizons.to_csv(audit / "paires_MMSE_avec_delai_reel.csv", index=False)

In [ ]:
import pandas as pd

candidats_1an = horizons.loc[
    horizons["delai_mmse_jours"].between(275, 455)
].copy()

candidats_1an["distance_a_365"] = (candidats_1an["delai_mmse_jours"] - 365).abs()

pilote_1an = (
    candidats_1an.sort_values([
        "PTID", "distance_a_365", "date_prediction",
        "date_mmse_fin", "id_irm_debut", "id_pet_debut"
    ])
    .drop_duplicates("PTID")
    .loc[:, [
        "PTID", "id_irm_debut", "date_irm_debut",
        "id_pet_debut", "date_pet_debut", "date_dx_debut", "DIAGNOSIS_debut",
        "date_prediction", "date_mmse_fin", "delai_mmse_jours", "mmse_fin"
    ]]
    .rename(columns={"mmse_fin": "cible_MMSE_1an"})
)

if len(pilote_1an) != 19 or pilote_1an["cible_MMSE_1an"].isna().any():
    raise ValueError("Le groupe obtenu ne correspond pas au bilan attendu.")

fichier = audit / "pilote_MMSE_1an_une_paire_par_participant.csv"
pilote_1an.to_csv(fichier, index=False, encoding="utf-8-sig")

print("Paires candidates :", len(candidats_1an), "(attendu : 20)")
print("Participants retenus :", len(pilote_1an), "(attendu : 19)")
print("Délai réel min / max :", pilote_1an["delai_mmse_jours"].min(),
      "/", pilote_1an["delai_mmse_jours"].max(), "jours")
print("Fichier enregistré :", fichier)

In [ ]:
import pandas as pd

# 1. Reprendre les visites avec IRM proposée, PET accessible et diagnostic.
departs = liens.copy()
departs["id_irm"] = departs["serie_mieux_notee_ADNI1"].fillna(
    departs["serie_choisie_Mayo_ADNI3"]
)
departs = departs[[
    "PTID", "date_irm", "id_irm", "date_pet", "id_pet", "date_dx", "DIAGNOSIS"
]].drop_duplicates()

# À cette date, les trois informations de départ sont disponibles.
departs["date_prediction"] = departs[["date_irm", "date_pet", "date_dx"]].max(axis=1)

# 2. Lire les MMSE de tous les participants, indépendamment des images au suivi.
tests_futurs = pd.read_csv(
    fichier_mmse, dtype="string", usecols=["PTID", "VISDATE", "MMSCORE"]
)
tests_futurs["PTID"] = tests_futurs["PTID"].str.strip()
tests_futurs["date_mmse"] = pd.to_datetime(tests_futurs["VISDATE"], errors="coerce")
tests_futurs["cible_MMSE_1an"] = pd.to_numeric(tests_futurs["MMSCORE"], errors="coerce")

tests_futurs = (
    tests_futurs.loc[
        tests_futurs["cible_MMSE_1an"].between(0, 30),
        ["PTID", "date_mmse", "cible_MMSE_1an"]
    ]
    .dropna()
    .drop_duplicates()
)
# Après suppression des doublons identiques, une date répétée a des scores différents.
tests_futurs = tests_futurs.loc[
    ~tests_futurs.duplicated(["PTID", "date_mmse"], keep=False)
]

# 3. Former toutes les associations temporelles admissibles, sans choisir par score.
options_1an = departs.merge(tests_futurs, on="PTID", how="inner")
options_1an["delai_jours"] = (
    options_1an["date_mmse"] - options_1an["date_prediction"]
).dt.days
options_1an = options_1an.loc[options_1an["delai_jours"].between(275, 455)].copy()

anciens = set(pilote_1an["PTID"])
actuels = set(options_1an["PTID"])

print("Visites de départ examinées :", len(departs[["PTID", "date_irm"]].drop_duplicates()))
print("Participants avec un MMSE futur dans la fenêtre :", len(actuels))
print("Dont participants du pilote initial :", len(actuels & anciens), "/ 19")
print("Participants supplémentaires :", len(actuels - anciens))
print("Participants du pilote initial non retrouvés :", len(anciens - actuels))

fichier = audit / "options_MMSE_1an_sans_images_de_suivi.csv"
options_1an.to_csv(fichier, index=False, encoding="utf-8-sig")

In [ ]:
import pandas as pd

options = options_1an.copy()
options["distance_a_365"] = (options["delai_jours"] - 365).abs()

# Ordre de priorité : délai proche d'un an, puis départ le plus ancien.
# Les dates et identifiants restants assurent un choix reproductible en cas d'égalité.
selection_1an = (
    options.sort_values([
        "PTID", "distance_a_365", "date_prediction", "date_irm",
        "date_pet", "date_dx", "date_mmse", "id_irm", "id_pet"
    ])
    .drop_duplicates("PTID")
    .copy()
)

# Conserver la trace de la règle qui a fourni l'IRM de départ.
sources = avec_proposition[["PTID", "date_irm", "source_proposition"]]
selection_1an = selection_1an.merge(
    sources, on=["PTID", "date_irm"], how="left", validate="many_to_one"
)

colonnes = [
    "PTID", "id_irm", "date_irm", "source_proposition",
    "id_pet", "date_pet", "date_dx", "DIAGNOSIS",
    "date_prediction", "date_mmse", "delai_jours", "cible_MMSE_1an"
]
selection_1an = selection_1an[colonnes]

if (len(selection_1an) != 112
    or not selection_1an["PTID"].is_unique
    or selection_1an[colonnes].isna().any().any()
    or not selection_1an["delai_jours"].between(275, 455).all()):
    raise ValueError("Le manifeste ne respecte pas les critères attendus.")

fichier = audit / "manifeste_MMSE_1an_112_participants.csv"
selection_1an.to_csv(fichier, index=False, encoding="utf-8-sig")

print("Participants distincts :", selection_1an["PTID"].nunique())
print("Délai réel min / médian / max (jours) :",
      selection_1an["delai_jours"].min(),
      selection_1an["delai_jours"].median(),
      selection_1an["delai_jours"].max())
print("Origine des propositions IRM :")
print(selection_1an["source_proposition"].value_counts().to_string())
print("Fichier enregistré :", fichier)

In [ ]:
from pathlib import Path
import pandas as pd

racine = Path(r"E:\Projet_Alzheimer\data")
manifeste = pd.read_csv(
    audit / "manifeste_MMSE_1an_112_participants.csv", dtype="string"
)

# Une ligne par image de départ : 112 IRM et 112 PET attendus.
images = pd.concat([
    manifeste[["PTID", "id_irm", "date_irm"]]
    .set_axis(["PTID", "id_image", "date_image"], axis=1)
    .assign(modalite="IRM"),
    manifeste[["PTID", "id_pet", "date_pet"]]
    .set_axis(["PTID", "id_image", "date_image"], axis=1)
    .assign(modalite="PET")
], ignore_index=True)

# Inventaires issus des contrôles DICOM précédents.
inv_irm = entetes[["id_commun", "patient_dicom", "date_dicom", "fichier_exemple"]].copy()
inv_irm.columns = ["id_image", "patient", "date_source", "dicom"]
inv_irm["PTID"] = inv_irm["patient"].str.extract(r"(\d{3}_S_\d{4})", expand=False)
inv_irm["date_image"] = pd.to_datetime(inv_irm["date_source"], format="%Y%m%d")
inv_irm["modalite"] = "IRM"

inv_pet = pet[["id_pet", "PTID", "date_pet", "exemple_dicom_pet"]].copy()
inv_pet.columns = ["id_image", "PTID", "date_image", "dicom"]
inv_pet["modalite"] = "PET"

cles = ["modalite", "id_image", "PTID", "date_image"]
inventaire = pd.concat([inv_irm[cles + ["dicom"]], inv_pet[cles + ["dicom"]]])
images["date_image"] = pd.to_datetime(images["date_image"]).dt.normalize()
inventaire["date_image"] = pd.to_datetime(inventaire["date_image"]).dt.normalize()

plan = images.merge(inventaire, on=cles, how="left", validate="one_to_one")
plan["dicom_accessible"] = plan["dicom"].map(
    lambda f: pd.notna(f) and Path(f).is_file()
)

# Réutiliser uniquement les fichiers marqués OK dans les bilans de conversion.
dossier_irm = racine / "06_IRM_NIfTI" / "premiere_paire_20261104_171451_793123"
dossier_pet = racine / "07_PET_NIfTI" / "premiere_paire_20261104_180032_856043"

b_irm = pd.read_csv(dossier_irm / "bilan_conversion.csv", dtype="string")
b_pet = pd.read_csv(dossier_pet / "bilan_conversion_PET.csv", dtype="string")

deja_convertis = {}
for _, ligne in b_irm.loc[b_irm["vérification"].eq("OK")].iterrows():
    deja_convertis[("IRM", ligne["IRM"])] = dossier_irm / ligne["fichier"]
for _, ligne in b_pet.loc[b_pet["verification"].eq("OK")].iterrows():
    deja_convertis[("PET", ligne["id_pet"])] = Path(ligne["fichier_nifti_pet"])

def nifti_existant(ligne):
    chemin = deja_convertis.get((ligne["modalite"], ligne["id_image"]))
    return str(chemin) if chemin is not None and chemin.is_file() else pd.NA

plan["nifti_verifie"] = plan.apply(nifti_existant, axis=1)
plan["action"] = "À convertir"
plan.loc[plan["nifti_verifie"].notna(), "action"] = "Déjà converti"
plan.loc[~plan["dicom_accessible"], "action"] = "Chemin DICOM à vérifier"

fichier_plan = audit / "plan_conversion_images_depart_112.csv"
plan.to_csv(fichier_plan, index=False, encoding="utf-8-sig")
display(pd.crosstab(plan["modalite"], plan["action"], margins=True))
print("Plan enregistré :", fichier_plan)

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import pydicom
import SimpleITK as sitk

# 1. Choisir un lot reproductible : les 3 premiers participants
# qui ont encore les deux modalités à convertir.
plan = pd.read_csv(fichier_plan, dtype="string")
restantes = plan.loc[plan["action"].eq("À convertir")].copy()
modalites = restantes.groupby("PTID")["modalite"].agg(set)
ids_lot = sorted(modalites[modalites.map(lambda x: x == {"IRM", "PET"})].index)[:3]
lot = restantes.loc[restantes["PTID"].isin(ids_lot)].sort_values(["PTID", "modalite"])

if len(ids_lot) != 3 or len(lot) != 6:
    raise ValueError("Le premier lot devrait contenir 3 participants et 6 images.")

journal = Path(audit) / "conversion_lot_001.csv"
colonnes = ["PTID", "modalite", "id_image", "statut", "nifti", "detail"]
suivi_lot = (pd.read_csv(journal, dtype="string").fillna("")
             if journal.exists() else pd.DataFrame(columns=colonnes))


def convertir_une_image(ligne, cible):
    exemple = Path(ligne.dicom)
    h = pydicom.dcmread(exemple, stop_before_pixels=True)

    # Contrôle de géométrie de la cellule 34.
    # Une série multiframe ou atypique est signalée pour traitement séparé.
    geo = verifier_volume(exemple)
    if geo["statut"] != "Géométrie régulière":
        raise ValueError(geo["statut"])

    fichiers = sitk.ImageSeriesReader.GetGDCMSeriesFileNames(
        str(exemple.parent), str(h.SeriesInstanceUID)
    )
    if len(fichiers) != geo["coupes"]:
        raise ValueError("Nombre de coupes différent entre les deux lectures.")
    if ligne.modalite == "PET" and len(fichiers) != 96:
        raise ValueError(f"PET : {len(fichiers)} coupes au lieu des 96 inventoriées.")

    lecteur = sitk.ImageSeriesReader()
    lecteur.SetFileNames(fichiers)
    if ligne.modalite == "PET":
        lecteur.SetOutputPixelType(sitk.sitkFloat64)
    volume = lecteur.Execute()

    if volume.GetDimension() != 3 or volume.GetSize()[2] != len(fichiers):
        raise ValueError(f"Dimensions inattendues : {volume.GetSize()}")

    # Le fichier ne prend son nom définitif qu'après vérification.
    temporaire = cible.with_name(cible.name.replace(".nii.gz", ".part.nii.gz"))
    try:
        sitk.WriteImage(volume, str(temporaire))
        relu = sitk.ReadImage(str(temporaire))
        geometrie_ok = (
            relu.GetSize() == volume.GetSize()
            and all(np.allclose(a, b, atol=1e-4, rtol=0) for a, b in [
                (relu.GetSpacing(), volume.GetSpacing()),
                (relu.GetOrigin(), volume.GetOrigin()),
                (relu.GetDirection(), volume.GetDirection())
            ])
        )
        pixels_ok = np.array_equal(
            sitk.GetArrayViewFromImage(relu), sitk.GetArrayViewFromImage(volume)
        )
        if not geometrie_ok or not pixels_ok:
            raise ValueError("Le NIfTI relu diffère du volume DICOM reconstruit.")
        temporaire.replace(cible)
    finally:
        temporaire.unlink(missing_ok=True)


for ligne in lot.itertuples(index=False):
    cle = (suivi_lot["modalite"] == ligne.modalite) & (suivi_lot["id_image"] == ligne.id_image)
    dossier_sortie = racine / (
        "06_IRM_NIfTI" if ligne.modalite == "IRM" else "07_PET_NIfTI"
    ) / "cohorte_112"
    dossier_sortie.mkdir(parents=True, exist_ok=True)
    cible = dossier_sortie / f"{ligne.modalite}_{ligne.id_image}.nii.gz"

    if (suivi_lot.loc[cle, "statut"] == "OK").any() and cible.is_file():
        print(ligne.modalite, ligne.id_image, ": déjà vérifié")
        continue

    try:
        convertir_une_image(ligne, cible)
        statut, detail = "OK", ""
    except Exception as erreur:
        statut, detail = "À vérifier", f"{type(erreur).__name__}: {erreur}"

    entree = [ligne.PTID, ligne.modalite, ligne.id_image, statut,
              str(cible) if statut == "OK" else "", detail]
    suivi_lot = pd.concat([suivi_lot.loc[~cle],
                           pd.DataFrame([entree], columns=colonnes)], ignore_index=True)
    tmp_csv = journal.with_suffix(".tmp")
    suivi_lot.to_csv(tmp_csv, index=False, encoding="utf-8-sig")
    tmp_csv.replace(journal)
    print(ligne.modalite, ligne.id_image, ":", statut, detail)

display(suivi_lot[["modalite", "id_image", "statut", "detail"]])

In [ ]:
from pathlib import Path
import warnings
import pandas as pd

plan = pd.read_csv(fichier_plan, dtype="string")
cles = ["modalite", "id_image"]

# Retirer les 2 volumes du pilote et les 6 conversions réussies du lot 001.
lot1 = pd.read_csv(Path(audit) / "conversion_lot_001.csv", dtype="string")
lot1_ok = lot1.loc[
    lot1["statut"].eq("OK")
    & lot1["nifti"].map(lambda f: pd.notna(f) and Path(f).is_file()), cles
]
travail = plan.loc[plan["action"].eq("À convertir")].merge(
    lot1_ok, on=cles, how="left", indicator=True, validate="one_to_one"
)
travail = travail.loc[travail["_merge"].eq("left_only")].sort_values(["PTID", "modalite"])

if len(travail) != 216:
    raise ValueError(f"{len(travail)} volumes au lieu de 216 : vérifier le plan et le lot 001.")

journal = Path(audit) / "conversion_suite_112.csv"
colonnes = ["PTID", "modalite", "id_image", "statut", "nifti", "detail", "avertissements"]
suivi = (pd.read_csv(journal, dtype="string").fillna("")
         if journal.exists() else pd.DataFrame(columns=colonnes))

for n, ligne in enumerate(travail.itertuples(index=False), start=1):
    cle = suivi["modalite"].eq(ligne.modalite) & suivi["id_image"].eq(ligne.id_image)
    precedent = suivi.loc[cle]

    if not precedent.empty:
        ancien = precedent.iloc[-1]
        if ancien["statut"] == "OK" and Path(ancien["nifti"]).is_file():
            continue
        if ancien["statut"] == "À vérifier":
            continue  # Garder ce cas pour examen, sans le retenter à chaque reprise.

    sous_dossier = "06_IRM_NIfTI" if ligne.modalite == "IRM" else "07_PET_NIfTI"
    cible = Path(racine) / sous_dossier / "cohorte_112" / f"{ligne.modalite}_{ligne.id_image}.nii.gz"
    cible.parent.mkdir(parents=True, exist_ok=True)

    with warnings.catch_warnings(record=True) as alertes:
        warnings.simplefilter("once")
        try:
            convertir_une_image(ligne, cible)
            statut, detail = "OK", ""
        except Exception as erreur:
            statut, detail = "À vérifier", f"{type(erreur).__name__} : {erreur}"

    messages = " | ".join(sorted({str(a.message) for a in alertes}))
    entree = [ligne.PTID, ligne.modalite, ligne.id_image, statut,
              str(cible) if statut == "OK" else "", detail, messages]
    suivi = pd.concat([suivi.loc[~cle], pd.DataFrame([entree], columns=colonnes)],
                      ignore_index=True)

    tmp = journal.with_suffix(".tmp")
    suivi.to_csv(tmp, index=False, encoding="utf-8-sig")
    tmp.replace(journal)

    if n % 10 == 0 or statut != "OK":
        print(f"{n}/216 · {ligne.modalite} {ligne.id_image} · {statut}", detail, flush=True)

print("\nBilan des 216 volumes restants :")
display(pd.crosstab(suivi["modalite"], suivi["statut"], margins=True))
print("Volumes avec avertissement :", suivi["avertissements"].ne("").sum())
print("Journal :", journal)

In [ ]:
from pathlib import Path
import pandas as pd

audit = Path(
    r"E:\Projet_Alzheimer\data\05_Manifestes"
    r"\cohorte_359_20261024_184212_495491"
    r"\audit_visites_IRM_20261104_143324_870635"
)

def lire(nom):
    return pd.read_csv(audit / nom, dtype="string")

participants = lire("manifeste_MMSE_1an_112_participants.csv")
plan = lire("plan_conversion_images_depart_112.csv")
lot1 = lire("conversion_lot_001.csv")
suite = lire("conversion_suite_112.csv")

# Retrouver les chemins vérifiés dans les trois sources.
cles = ["PTID", "modalite", "id_image"]
pilote = (
    plan.loc[plan["action"].eq("Déjà converti"), cles + ["nifti_verifie"]]
    .rename(columns={"nifti_verifie": "nifti"})
)
lots = pd.concat([lot1, suite], ignore_index=True)
volumes = pd.concat(
    [pilote, lots.loc[lots["statut"].eq("OK"), cles + ["nifti"]]],
    ignore_index=True
)

if len(volumes) != 224 or volumes.duplicated(cles).any():
    raise ValueError("Le nombre de volumes ou l’unicité des identifiants est à vérifier.")

# Associer les chemins par participant ET par identifiant d’image.
base = participants.copy()
for modalite, identifiant, chemin in [
    ("IRM", "id_irm", "chemin_irm"),
    ("PET", "id_pet", "chemin_pet"),
]:
    source = (
        volumes.loc[volumes["modalite"].eq(modalite), ["PTID", "id_image", "nifti"]]
        .rename(columns={"id_image": identifiant, "nifti": chemin})
    )
    base = base.merge(source, on=["PTID", identifiant], how="left", validate="one_to_one")

if len(base) != 112 or not base["PTID"].is_unique:
    raise ValueError("Le tableau doit contenir une seule ligne par participant.")

for chemin in ["chemin_irm", "chemin_pet"]:
    accessible = base[chemin].map(lambda p: pd.notna(p) and Path(p).is_file())
    if not accessible.all():
        raise ValueError(f"{(~accessible).sum()} fichier(s) à vérifier dans {chemin}.")

cible = pd.to_numeric(base["cible_MMSE_1an"], errors="coerce")
if not cible.between(0, 30).all():
    raise ValueError("Au moins un score MMSE futur est absent ou invalide.")

fichier_base = audit / "base_MMSE_1an_112_avec_chemins.csv"
base.to_csv(fichier_base, index=False, encoding="utf-8-sig")
print("Participants avec IRM + PET + MMSE futur :", len(base))
print("Tableau enregistré :", fichier_base)

# Regrouper les messages qui ne diffèrent que par la valeur de l’identifiant.
alertes = suite["avertissements"].dropna().str.strip()
alertes = alertes[alertes.ne("")]
types_alertes = alertes.str.replace(r"\b\d+(?:\.\d+){2,}\b", "<UID>", regex=True)
print("\nAvertissements du dernier lot : nombre de volumes par message")
print(types_alertes.value_counts().to_string())